<a href="https://colab.research.google.com/github/YuriArduino/Estudos_Artificial_Intelligence/blob/Lang_chain/Lang_chain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Aula 1: RAG

In [ ]:
!python -m pip -q install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 28.4 MB/s eta 0:00:00


In [ ]:
!pip install -q --upgrade langchain langchain-google-genai pypdf langchain-community

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.9.0 which is incompatible.


In [ ]:
import langchain
print(langchain.__version__)

1.0.5


In [ ]:
import logging
from datetime import datetime
import pytz  # biblioteca já disponível no Colab

# Define o fuso horário de Brasília (UTC-3)
brasilia_tz = pytz.timezone("America/Sao_Paulo")

class TZFormatter(logging.Formatter):
    def formatTime(self, record, datefmt=None):
        dt = datetime.fromtimestamp(record.created, tz=brasilia_tz)
        return dt.strftime(datefmt or "%H:%M:%S")

# Configuração global de logs
handler = logging.StreamHandler()
handler.setFormatter(TZFormatter("%(asctime)s | %(levelname)-7s | %(message)s"))

logging.basicConfig(
    level=logging.INFO,
    handlers=[handler],
    force=True
)

logger = logging.getLogger(__name__)
logger.info("Logging configurado com Sucesso - Horário de Brasília (UTC-3).")

22:52:20 | INFO    | Logging configurado com Sucesso - Horário de Brasília (UTC-3).


In [ ]:
import os
import logging
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

logger = logging.getLogger(__name__)

class LlmmodelLoader:
    """
    Carrega e configura um modelo LLM usando a API do Google Gemini.
    """

    def carregar_modelos(self, model: str = "gemini-2.5-flash", temperature: float = 0):
        try:
            logger.info("--- Carregando modelos ---")

            # Obtém e valida a chave da API
            api_key = userdata.get("GEMINI_API_KEY")
            if not api_key:
                raise RuntimeError("Chave GEMINI_API_KEY não encontrada no userdata do Colab.")
            os.environ["GOOGLE_API_KEY"] = api_key

            # Inicializa o modelo
            llm = ChatGoogleGenerativeAI(model=model, temperature=temperature)

            logger.info(f"✅ Modelo '{model}' configurado com sucesso!")
            return llm

        except Exception as e:
            logger.error(f"❌ Erro na configuração do modelo: {e}")
            return None


if __name__ == "__main__":
    loader = LlmmodelLoader()
    llm_model = loader.carregar_modelos()

    if llm_model:
        logger.info("Modelo LLM carregado com sucesso.")
    else:
        logger.error("Falha ao carregar o modelo LLM.")

22:52:41 | INFO    | --- Carregando modelos ---
22:52:42 | INFO    | ✅ Modelo 'gemini-2.5-flash' configurado com sucesso!
22:52:42 | INFO    | Modelo LLM carregado com sucesso.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# EXEMPLO 1: Prompting Tradicional (sem RAG)

pergunta = "Qual é a política de home office da nossa empresa?"

prompt_tradicional = ChatPromptTemplate.from_template(
    "Responda a seguinte pergunta: {pergunta}"
)

print(prompt_tradicional.format(pergunta=pergunta))


Human: Responda a seguinte pergunta: Qual é a política de home office da nossa empresa?


In [ ]:
chain_tradicional = prompt_tradicional | llm_model

resposta_tradicional = chain_tradicional.invoke({"pergunta": pergunta})

In [ ]:
print(resposta_tradicional.content)

Como sou uma inteligência artificial e não tenho acesso às políticas internas específicas da sua empresa, não consigo te dar a resposta exata.

No entanto, posso te explicar os tipos comuns de políticas de home office que as empresas adotam hoje em dia, para que você tenha uma ideia do que procurar:

1.  **Totalmente Remoto (Full Remote):** Todos os colaboradores trabalham de casa permanentemente, independentemente da função. A empresa pode ter um escritório físico, mas ele não é o local principal de trabalho.
2.  **Híbrido (Hybrid):** Esta é a modalidade mais comum atualmente e pode ter várias variações:
    *   **Dias Fixos:** Os colaboradores têm dias específicos da semana para ir ao escritório (ex: terças e quintas) e os demais dias trabalham de casa.
    *   **Dias Flexíveis:** Os colaboradores precisam cumprir um número mínimo de dias no escritório por mês ou semana, mas podem escolher quais dias.
    *   **Baseado em Equipe:** Cada equipe ou departamento define sua própria polít

In [ ]:
import logging
import time
from pathlib import Path
from typing import List
from urllib.parse import unquote

import requests

logger = logging.getLogger(__name__)


class DocumentLoader:
    """
    Carrega documentos de URLs e grava-os em uma pasta persistente (por padrão /content/tmp).
    Ajusta os metadados 'source' usando o nome original do arquivo.
    """

    def __init__(self, timeout: int = 20, persist_dir: str = "/content/tmp"):
        self.timeout = timeout
        self.persist_dir = Path(persist_dir)
        self.persist_dir.mkdir(parents=True, exist_ok=True)

    @staticmethod
    def _extrair_nome_arquivo(url: str) -> str:
        """Extrai o nome do arquivo de uma URL, decodificando caracteres especiais."""
        return unquote(url.split("/")[-1]) or f"download_{int(time.time())}.dat"

    def _gerar_caminho_unico(self, nome_arquivo: str) -> Path:
        """Gera um caminho único dentro de self.persist_dir sem sobrescrever existente."""
        destino = self.persist_dir / nome_arquivo
        if not destino.exists():
            return destino
        # se já existe, adiciona timestamp antes da extensão
        stem = destino.stem
        suffix = destino.suffix or ""
        novo_nome = f"{stem}_{int(time.time())}{suffix}"
        return self.persist_dir / novo_nome

    def _baixar(self, url: str) -> Path:
        """
        Baixa um arquivo de uma URL e salva em self.persist_dir.
        Retorna o caminho do arquivo gravado.
        """
        dl_url = (
            url + "?raw=true"
            if "github.com" in url and "?raw=true" not in url
            else url
        )

        logger.debug(f"Iniciando download de: {dl_url}")
        response = requests.get(dl_url, timeout=self.timeout)
        response.raise_for_status()

        nome_arquivo = self._extrair_nome_arquivo(url)
        caminho_final = self._gerar_caminho_unico(nome_arquivo)

        # grava bytes no caminho final
        caminho_final.write_bytes(response.content)
        logger.debug(f"Arquivo gravado em: {caminho_final}")

        return caminho_final

    def carregar(self, urls: List[str]) -> List[Path]:
        """
        Baixa e armazena os arquivos (persistentes) em self.persist_dir.
        Retorna a lista de caminhos dos arquivos criados.
        """
        logger.info("Iniciando o carregamento de documentos...")
        arquivos: List[Path] = []

        for url in urls:
            nome = self._extrair_nome_arquivo(url)
            try:
                caminho = self._baixar(url)
                arquivos.append(caminho)
                logger.info(f"✅ '{nome}' carregado com sucesso em: {caminho}")
            except Exception as e:
                logger.error(f"❌ Erro ao carregar '{nome}': {e}")

        logger.info("Processo concluído.")
        return arquivos

urls = [
    "https://github.com/YuriArduino/Estudos_Artificial_Intelligence/blob/Dados/politica_home_office.pdf"
]

loader = DocumentLoader(timeout=20, persist_dir="/content/tmp")
arquivos_baixados = loader.carregar(urls)

logger.info("Arquivos baixados:")
for arquivo in arquivos_baixados:
  logger.info(f"  - {arquivo}")

22:52:51 | INFO    | Iniciando o carregamento de documentos...
22:52:52 | INFO    | ✅ 'politica_home_office.pdf' carregado com sucesso em: /content/tmp/politica_home_office.pdf
22:52:52 | INFO    | Processo concluído.
22:52:52 | INFO    | Arquivos baixados:
22:52:52 | INFO    |   - /content/tmp/politica_home_office.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import logging

logger = logging.getLogger(__name__)

loader = PyPDFLoader("/content/tmp/politica_home_office.pdf") # Using the file path without the timestamp

documento = loader.load()

logger.info(f"✅ Documento carregado")

22:52:54 | INFO    | NumExpr defaulting to 2 threads.
22:53:16 | INFO    | TensorFlow version 2.19.0 available.
22:53:16 | INFO    | JAX version 0.7.2 available.
22:53:20 | INFO    | ✅ Documento carregado


In [ ]:
documento

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-07-07T09:42:25-03:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-07-07T09:42:25-03:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '/content/tmp/politica_home_office.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content="Política de Trabalho Remoto e Híbrido\nVersão 2.1 - Atualizada em Janeiro 2024\n1. Objetivo\nEsta política estabelece as diretrizes para o trabalho remoto e híbrido na Empresa XYZ, visando\npromover a flexibilidade, o bem-estar dos funcionários e a manutenção da produtividade e\ncolaboração.\n2. Elegibilidade\nTodos os funcionários em tempo integral, que completaram o período de experiência de 90 dias e\ncujas funções são compatíveis com o trabalho remoto, são elegíveis para aderir ao modelo de trabalho\nhíbrido. A aprovação final está sujeita ao acordo com o gestor direto.\n

In [ ]:
contexto_empresa = documento[0].page_content

In [ ]:
contexto_empresa

"Política de Trabalho Remoto e Híbrido\nVersão 2.1 - Atualizada em Janeiro 2024\n1. Objetivo\nEsta política estabelece as diretrizes para o trabalho remoto e híbrido na Empresa XYZ, visando\npromover a flexibilidade, o bem-estar dos funcionários e a manutenção da produtividade e\ncolaboração.\n2. Elegibilidade\nTodos os funcionários em tempo integral, que completaram o período de experiência de 90 dias e\ncujas funções são compatíveis com o trabalho remoto, são elegíveis para aderir ao modelo de trabalho\nhíbrido. A aprovação final está sujeita ao acordo com o gestor direto.\n3. Modalidade e Horário\n3.1. Modelo Híbrido: A modalidade padrão é híbrida, compreendendo 3 (três) dias de trabalho\nremoto (home office) e 2 (dois) dias de trabalho presencial no escritório, por semana.\n3.2. Dias Presenciais: Os dias de trabalho presencial serão definidos em comum acordo entre a\nequipe e o gestor, priorizando as terças-feiras para reuniões de alinhamento geral da equipe.\n3.3. Horário Flexível

In [ ]:
print(contexto_empresa[:500] + "...")

Política de Trabalho Remoto e Híbrido
Versão 2.1 - Atualizada em Janeiro 2024
1. Objetivo
Esta política estabelece as diretrizes para o trabalho remoto e híbrido na Empresa XYZ, visando
promover a flexibilidade, o bem-estar dos funcionários e a manutenção da produtividade e
colaboração.
2. Elegibilidade
Todos os funcionários em tempo integral, que completaram o período de experiência de 90 dias e
cujas funções são compatíveis com o trabalho remoto, são elegíveis para aderir ao modelo de trabalho...


In [ ]:
# Exemplo 2: Com RAG - Usando o contexto do PDF

prompt_rag = ChatPromptTemplate.from_template("""
Use o contexto abaixo para responder a pergunta.
Se não souber a resposta baseado no contexto, diga que não tem a informação.

Contexto: {contexto}
Pergunta: {pergunta}

RespostaResposta:""")

In [ ]:
chain_rag = prompt_rag | llm_model

resposta_rag = chain_rag.invoke({"contexto": contexto_empresa, "pergunta": pergunta})

In [ ]:
print(resposta_rag.content)

A política de trabalho da empresa é **híbrida**, não exclusivamente de home office. Ela estabelece o seguinte:

*   **Modalidade Padrão:** 3 (três) dias de trabalho remoto (home office) e 2 (dois) dias de trabalho presencial no escritório, por semana.
*   **Elegibilidade:** Todos os funcionários em tempo integral que completaram o período de experiência de 90 dias, cujas funções são compatíveis com o trabalho remoto, e com aprovação do gestor direto.
*   **Dias Presenciais:** Definidos em comum acordo entre a equipe e o gestor, sendo as **terças-feiras o dia presencial obrigatório** para reuniões de alinhamento geral da equipe e reuniões estratégicas.
*   **Horário:** A jornada de trabalho de 8 horas diárias pode ser cumprida com flexibilidade, iniciando entre 07:00 e 10:00. O horário de 'core time', no qual todos devem estar disponíveis online ou no escritório, é das 10:00 às 16:00.
*   **Equipamentos e Auxílio de Custo:** A Empresa XYZ fornecerá notebook e, mediante solicitação e apr

# Aula 2: Embeddings

In [ ]:
!pip install -q --upgrade faiss-cpu chromadb langchain-pinecone pinecone-client

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.4 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.3.4 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-exporter-otlp-proto-common==1.37.0, but you have opentelemetry-exporter-otlp-proto-common 1.38.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-proto==1.37.0, but you have opentelemetry-proto 1.38.0 which is incompatible.
opentelemetry-exporter-otlp-

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")


In [ ]:
embeddings.embed_query("Política de home office da empresa")

[0.016149114817380905,
 0.040602732449769974,
 0.03302507847547531,
 -0.059951938688755035,
 0.0015303720720112324,
 -0.00802367553114891,
 -0.0035909658763557673,
 0.0012388312024995685,
 0.017689721658825874,
 -0.03018672950565815,
 0.0025397443678230047,
 -0.019532717764377594,
 -0.009552416391670704,
 0.023326490074396133,
 0.1223549023270607,
 0.010480530560016632,
 0.025290023535490036,
 0.019415486603975296,
 -0.012007049284875393,
 -0.002410824876278639,
 0.020195970311760902,
 -0.001310814986936748,
 0.0013476797612383962,
 -0.022229310125112534,
 -0.017976293340325356,
 -0.05222192406654358,
 0.005175173748284578,
 0.005886222701519728,
 0.027025695890188217,
 0.0054599749855697155,
 -0.011313029564917088,
 0.023075412958860397,
 -0.00015346195141319185,
 0.012506196275353432,
 0.007758465129882097,
 0.012851455248892307,
 0.02055903896689415,
 0.007812640629708767,
 -0.007826380431652069,
 0.018867721781134605,
 -0.019412031397223473,
 -0.0011542309075593948,
 0.003912106156

In [ ]:
documentos_empresa = [
    Document(
        page_content="Política de férias: Funcionários têm direito a 30 dias de férias após 12 meses. A solicitação deve ser feita com 30 dias de antecedência.",
        metadata={"tipo": "política", "departamento": "RH", "ano": 2024, "id_doc": "doc001"}
    ),

    Document(
        page_content="Processo de reembolso de despesas: Envie a nota fiscal pelo portal financeiro. O reembolso ocorre em até 5 dias úteis.",
        metadata={"tipo": "processo", "departamento": "Financeiro", "ano": 2023, "id_doc": "doc002"}
    ),

    Document(
        page_content="Guia de TI: Para configurar a VPN, acesse vpn.nossaempresa.com e siga as instruções para seu sistema operacional.",
        metadata={"tipo": "tutorial", "departamento": "TI", "ano": 2024, "id_doc": "doc003"}
    ),

    Document(
        page_content="Código de Ética e Conduta: Valorizamos o respeito, a integridade e a colaboração. Casos de assédio não serão tolerados.",
        metadata={"tipo": "política", "departamento": "RH", "ano": 2022, "id_doc": "doc004"}
    )

]

In [ ]:
from langchain_community.vectorstores import FAISS
import faiss

d = 768 #768 dimensões por padrão
index_hnsw = faiss.IndexHNSWFlat(d, 32)

22:53:54 | INFO    | Loading faiss with AVX2 support.
22:53:54 | INFO    | Successfully loaded faiss with AVX2 support.


In [ ]:
faiss_db = FAISS.from_documents(documentos_empresa, embeddings)

pergunta = "Como peço minhas férias?"
resultados = faiss_db.similarity_search(pergunta, k=2)

In [ ]:
print(f"\n🔍 Pergunta: '{pergunta}'")
print("\n Documentos mais relevantes (FAISS):")
for doc in resultados:
    print(f"- {doc.page_content}")
    print(f" (Metadados: {doc.metadata})")


🔍 Pergunta: 'Como peço minhas férias?'

 Documentos mais relevantes (FAISS):
- Política de férias: Funcionários têm direito a 30 dias de férias após 12 meses. A solicitação deve ser feita com 30 dias de antecedência.
 (Metadados: {'tipo': 'política', 'departamento': 'RH', 'ano': 2024, 'id_doc': 'doc001'})
- Processo de reembolso de despesas: Envie a nota fiscal pelo portal financeiro. O reembolso ocorre em até 5 dias úteis.
 (Metadados: {'tipo': 'processo', 'departamento': 'Financeiro', 'ano': 2023, 'id_doc': 'doc002'})


In [ ]:
import shutil
if os.path.exists("./chroma_db_persist"):
    shutil.rmtree("./chroma_db_persist")
print("Diretório ChromaDB limpo. Tente executar novamente.")

Diretório ChromaDB limpo. Tente executar novamente.


In [ ]:
from langchain_community.vectorstores import Chroma

chroma_db = Chroma.from_documents(
    documents=documentos_empresa,
    embedding=embeddings,
    # Adicionar persist_directory se quiser persistir os dados
    persist_directory="./chroma_db_persist",
    collection_metadata={"hnsw:space": "cosine"} # Opcional: especificar a métrica de similaridade
)

In [ ]:
resultados = chroma_db.similarity_search(pergunta, k=2)

for doc in resultados:
  print(f"- {doc.page_content}")

- Política de férias: Funcionários têm direito a 30 dias de férias após 12 meses. A solicitação deve ser feita com 30 dias de antecedência.
- Processo de reembolso de despesas: Envie a nota fiscal pelo portal financeiro. O reembolso ocorre em até 5 dias úteis.


In [ ]:
pergunta_rh = "Quais são as regras da empresa?"

resultados_filtrados = chroma_db.similarity_search(
    pergunta_rh,
    k=2,
    filter={"$and": [{"departamento": "RH"}, {"tipo": "política"}]}
    )

In [ ]:
print(f"\n Pergunta: '{pergunta_rh}' com filtro para políticas de RH")
print("\n Documentos relevantes e filtrados (Chroma):")

for doc in resultados_filtrados:
  print(f"- {doc.page_content}")
  print(f"  (Departamento: {doc.metadata['departamento']}, Tipo: {doc.metadata['tipo']})")


 Pergunta: 'Quais são as regras da empresa?' com filtro para políticas de RH

 Documentos relevantes e filtrados (Chroma):
- Código de Ética e Conduta: Valorizamos o respeito, a integridade e a colaboração. Casos de assédio não serão tolerados.
  (Departamento: RH, Tipo: política)
- Política de férias: Funcionários têm direito a 30 dias de férias após 12 meses. A solicitação deve ser feita com 30 dias de antecedência.
  (Departamento: RH, Tipo: política)


---

## Para saber mais **HNSW (Hierarchical Navigable Small World)**

## **1. Estrutura do HNSW**

O HNSW é um índice baseado em **grafos** que organiza vetores de forma hierárquica.

* Cada nó do grafo representa um item do conjunto de dados.
* Cada nó mantém ligações com seus vizinhos mais próximos.

Essa estrutura possibilita **saltos estratégicos** durante a busca, acelerando a recuperação dos itens mais semelhantes — ainda que não se atinja a mesma precisão de uma busca exaustiva.

---

## **2. Papel do Número de Vizinhos**

Um dos parâmetros essenciais na configuração do HNSW é o número de vizinhos conectados a cada nó (*geralmente definido como 32*). Esse parâmetro afeta diretamente:

* **Qualidade da busca**

  * Um número maior de vizinhos tende a aumentar o *recall* (probabilidade de encontrar itens realmente próximos ao vetor de consulta).
  * Especialmente útil em dados com **alta variabilidade**.

* **Desempenho e custo computacional**

  * Mais conexões aumentam o uso de memória e o tempo de construção do índice.
  * Valores muito altos podem gerar lentidão em ambientes com **recursos limitados**.

---

## **3. Considerações na Escolha do Valor**

A definição do número de vizinhos é um **trade-off entre velocidade e acurácia**:

* Projetos que priorizam **precisão** e dispõem de **recursos robustos** → usar valores mais altos.
* Cenários com **grandes volumes de dados** ou **protótipos iniciais** → valores menores oferecem respostas mais rápidas, embora com leve perda de precisão.

A escolha ideal exige **testes práticos** e aferição de métricas de similaridade, sempre alinhada aos objetivos do projeto.

---

## **4. Exemplo Prático em Código**

```python
import faiss

# Dimensão dos vetores
d = 768

# Número de vizinhos configurados para o índice HNSW
M = 32

# Criação do índice HNSW
index = faiss.IndexHNSWFlat(d, M)
```

Nesse exemplo:

* Vetores de dimensão **768**.
* Cada nó mantém **32 conexões**.
* Essa configuração serve como **ponto de partida** e pode ser ajustada conforme a avaliação de desempenho e a natureza dos dados.

---

## **5. Conclusão**

A experimentação com diferentes valores de vizinhos é crucial para encontrar o **equilíbrio ideal**:

* **Buscas rápidas**.
* **Resultados relevantes**.
* **Uso eficiente de recursos**.

---



In [ ]:
import os
from google.colab import userdata

os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

In [ ]:
# Validação mínima model Pydantic v2
import os
from pydantic import Field, ValidationError, field_validator
from pydantic_settings import BaseSettings # Importar BaseSettings de pydantic_settings

logger = logging.getLogger(__name__) # Obter o logger

class PineconeSettings(BaseSettings):
    pinecone_api_key: str = Field(..., env="PINECONE_API_KEY")
    index_name: str = Field("langchain-rag", env="PINECONE_INDEX_NAME")
    cloud: str = Field("aws", env="PINECONE_CLOUD")
    region: str = Field("us-east-1", env="PINECONE_REGION")

    model_config = {
        "env_prefix": "",    # usa os nomes exatos das vars (sem prefixo)
        "extra": "ignore",   # não falha se houver outras vars no ambiente
    }

    @field_validator("pinecone_api_key")
    @classmethod
    def check_key_not_empty(cls, v: str) -> str:
        if not v or not v.strip():
            raise ValueError("PINECONE_API_KEY não pode ser vazio")
        return v

try:
    settings = PineconeSettings()
except ValidationError as e:
    # Se falhar, isso facilita debug rápido no Colab
    logger.error(f"Erro nas configurações do Pinecone:\n {e}") # Usando logger.error
    raise

In [ ]:
from pinecone import Pinecone as PineconeClient, ServerlessSpec
from langchain_pinecone import Pinecone  # mantenha se for usar LangChain depois

logger = logging.getLogger(__name__)

pinecone_client = PineconeClient(api_key=settings.pinecone_api_key)
spec = ServerlessSpec(cloud=settings.cloud, region=settings.region)
index_name = settings.index_name

logger.info(f"Pinecone client e spec criados — index_name: {index_name}") # Alterado para logger.info

22:53:58 | INFO    | Pinecone client e spec criados — index_name: langchain-rag


In [ ]:
# Verificando se o índice existe. Se não, ele será criado.
# Nome do seu índice no Pinecone
index_name = "langchain-rag"
# A dimensão correta para os embeddings do Gemini é 3072
pinecone_dimension = 3072

if index_name not in pinecone_client.list_indexes().names():
    print(f"Índice '{index_name}' não encontrado. Criando...")
    pinecone_client.create_index(
        name=index_name,
        dimension=pinecone_dimension,  # Usando a dimensão correta de 3072
        metric="cosine",
        spec=spec
    )
    print(f"Índice '{index_name}' criado no Pinecone com dimensão {pinecone_dimension}.")

    # Esperar o índice ficar pronto (opcional, mas recomendado)
    while not pinecone_client.describe_index(index_name).status.ready:
        time.sleep(1)
    print(f"Índice '{index_name}' está pronto.")


    pinecone_db = Pinecone.from_documents(
        documentos_empresa,
        embeddings, # embeddings é o objeto do Gemini, que tem 3072 dimensões
        index_name=index_name
    )
    print(f"Documentos adicionados ao índice '{index_name}'.")

else:
    print(f"Conectando ao índice existente '{index_name}'.")
    # Verificar se a dimensão do índice existente corresponde à do embedding
    existing_index_info = pinecone_client.describe_index(index_name)
    if existing_index_info.dimension != pinecone_dimension:
        print(f"A dimensão do índice existente ({existing_index_info.dimension}) não corresponde à dimensão esperada ({pinecone_dimension}).")
        print(f"Excluindo índice existente '{index_name}' e recriando-o.")
        pinecone_client.delete_index(index_name)
        # Esperar a exclusão ser concluída (opcional)
        while index_name in pinecone_client.list_indexes().names():
             time.sleep(1)
        print(f"Índice '{index_name}' excluído.")
        # Recriar o índice
        print(f"Criando índice '{index_name}' novamente com dimensão {pinecone_dimension}.")
        pinecone_client.create_index(
            name=index_name,
            dimension=pinecone_dimension,
            metric="cosine",
            spec=spec
        )
        # Esperar o índice ficar pronto
        while not pinecone_client.describe_index(index_name).status.ready:
             time.sleep(1)
        print(f"Índice '{index_name}' está pronto.")
        # Adicionar documentos ao novo índice
        pinecone_db = Pinecone.from_documents(
            documentos_empresa,
            embeddings,
            index_name=index_name
        )
        print(f"Documentos adicionados ao índice '{index_name}'.")

    else:
        print(f"A dimensão do índice existente ({existing_index_info.dimension}) corresponde à esperada ({pinecone_dimension}). Conectando...")
        # Se o índice já existe com a dimensão correta, apenas conectamos
        pinecone_db = Pinecone.from_existing_index(
            index_name,
            embeddings
        )
        print(f"Conectado ao índice existente '{index_name}'.")

Conectando ao índice existente 'langchain-rag'.
A dimensão do índice existente (3072) corresponde à esperada (3072). Conectando...
Conectado ao índice existente 'langchain-rag'.


In [ ]:
if pinecone_db:
    # Busca por similaridade
    pergunta_ti = "Como configuro a VPN?"
    resultados_pinecone = pinecone_db.similarity_search(pergunta_ti, k=2)

    print(f"\n🔍 Pergunta: '{pergunta_ti}'")
    print("\n📄 Documentos mais relevantes (Pinecone):")
    for doc in resultados_pinecone:
        print(f"- {doc.page_content}")
        print(f"  (Metadados: {doc.metadata})")

    # Busca com filtro (Pinecone também suporta!)
    resultados_pinecone_filtrados = pinecone_db.similarity_search(
        "informações sobre regras",
        k=2,
        filter={"tipo": "política"}
    )
    print(f"\n🔍 Pergunta: 'informações sobre regras' com filtro para tipo='política'")
    print("\n📄 Documentos relevantes e filtrados (Pinecone):")
    for doc in resultados_pinecone_filtrados:
        print(f"- {doc.page_content}")
        print(f"  (Tipo: {doc.metadata['tipo']})")


🔍 Pergunta: 'Como configuro a VPN?'

📄 Documentos mais relevantes (Pinecone):
- Guia de TI: Para configurar a VPN, acesse vpn.nossaempresa.com e siga as instruções para seu sistema operacional.
  (Metadados: {'ano': 2024.0, 'departamento': 'TI', 'id_doc': 'doc003', 'tipo': 'tutorial'})
- Código de Ética e Conduta: Valorizamos o respeito, a integridade e a colaboração. Casos de assédio não serão tolerados.
  (Metadados: {'ano': 2022.0, 'departamento': 'RH', 'id_doc': 'doc004', 'tipo': 'política'})

🔍 Pergunta: 'informações sobre regras' com filtro para tipo='política'

📄 Documentos relevantes e filtrados (Pinecone):
- Código de Ética e Conduta: Valorizamos o respeito, a integridade e a colaboração. Casos de assédio não serão tolerados.
  (Tipo: política)
- Política de férias: Funcionários têm direito a 30 dias de férias após 12 meses. A solicitação deve ser feita com 30 dias de antecedência.
  (Tipo: política)


#Aula 3: Embeddings de Alta Performance

In [ ]:
!pip install -q langchain langchain-google-genai sentence-transformers scikit-learn langchain-community

In [ ]:
import numpy as np

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
import time

In [ ]:
# Textos de exemplo para nosso teste
textos_teste = [
    "Qual é a política de férias da nossa empresa?",
    "Preciso de um relatório de despesas de viagem.",
    "Como configuro o acesso à rede privada virtual (VPN)?",
    "Onde encontro o código de conduta da organização?",
    "Quero entender o processo de avaliação de performance."
]

In [ ]:
## Google Gemini Embeddings

gemini_embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

start_time = time.time()
embeddings_gemini = gemini_embeddings.embed_documents(textos_teste)
end_time = time.time()

_tempo_total = end_time - start_time
_tempo_medio = _tempo_total / max(1, len(textos_teste))

meta = {
    "model": "gemini-embedding-001",
    "n_docs": len(textos_teste),
    "tempo_total_s": _tempo_total,
    "tempo_medio_por_doc_s": _tempo_medio,
    "dimensao_vetor": len(embeddings_gemini[0]) if embeddings_gemini else None
}

meta_summary = (
    f"model={meta['model']} | n_docs={meta['n_docs']} | "
    f"tempo_total_s={meta['tempo_total_s']:.4f} | tempo_medio_por_doc_s={meta['tempo_medio_por_doc_s']:.4f} | "
    f"dimensao_vetor={meta['dimensao_vetor']}"
)

logger.info(f"Tempo de processamento (gemini-embedding-001): {end_time - start_time} segundos")
logger.info(f"  - Dimensões do vetor (gemini-embedding-001): {len(embeddings_gemini[0])}")


22:54:19 | INFO    | Tempo de processamento (gemini-embedding-001): 0.35669565200805664 segundos
22:54:19 | INFO    |   - Dimensões do vetor (gemini-embedding-001): 3072


In [ ]:
## intfloat/multilingual-e5-large

multilingual_e5_large_embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")
start_time = time.time()
embeddings_multilingual_e5_large = multilingual_e5_large_embeddings.embed_documents(textos_teste)
end_time = time.time()

_tempo_total = end_time - start_time
_tempo_medio = _tempo_total / max(1, len(textos_teste))

meta_multilingual_e5 = {
    "model": "intfloat/multilingual-e5-large",
    "n_docs": len(textos_teste),
    "tempo_total_s": _tempo_total,
    "tempo_medio_por_doc_s": _tempo_medio,
    "dimensao_vetor": len(embeddings_multilingual_e5_large[0]) if embeddings_multilingual_e5_large else None
}

meta_summary_multilingual_e5 = (
    f"model={meta_multilingual_e5['model']} | n_docs={meta_multilingual_e5['n_docs']} | "
    f"tempo_total_s={meta_multilingual_e5['tempo_total_s']:.4f} | tempo_medio_por_doc_s={meta_multilingual_e5['tempo_medio_por_doc_s']:.4f} | "
    f"dimensao_vetor={meta_multilingual_e5['dimensao_vetor']}"
)

logger.info(f"Tempo de processamento (intfloat/multilingual-e5-large): {end_time - start_time} segundos")
logger.info(f"  - Dimensões do vetor (intfloat/multilingual-e5-large): {len(embeddings_multilingual_e5_large[0])}")


/tmp/ipython-input-947692835.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  multilingual_e5_large_embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")
22:54:19 | INFO    | Use pytorch device_name: cuda:0
22:54:19 | INFO    | Load pretrained SentenceTransformer: intfloat/multilingual-e5-large
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse 

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

22:55:25 | INFO    | Tempo de processamento (intfloat/multilingual-e5-large): 1.742659568786621 segundos
22:55:25 | INFO    |   - Dimensões do vetor (intfloat/multilingual-e5-large): 1024


In [ ]:
## all-MiniLM-L6-v2

minilm_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
start_time = time.time()
embeddings_minilm = minilm_embeddings.embed_documents(textos_teste)
end_time = time.time()

_tempo_total = end_time - start_time
_tempo_medio = _tempo_total / max(1, len(textos_teste))

meta_minilm = {
    "model": "all-MiniLM-L6-v2",
    "n_docs": len(textos_teste),
    "tempo_total_s": _tempo_total,
    "tempo_medio_por_doc_s": _tempo_medio,
    "dimensao_vetor": len(embeddings_minilm[0]) if embeddings_minilm else None
}

meta_summary_minilm = (
    f"model={meta_minilm['model']} | n_docs={meta_minilm['n_docs']} | "
    f"tempo_total_s={meta_minilm['tempo_total_s']:.4f} | tempo_medio_por_doc_s={meta_minilm['tempo_medio_por_doc_s']:.4f} | "
    f"dimensao_vetor={meta_minilm['dimensao_vetor']}"
)

logger.info(f"Tempo de processamento (all-MiniLM-L6-v2): {end_time - start_time} segundos")
logger.info(f"  - Dimensões do vetor (all-MiniLM-L6-v2): {len(embeddings_minilm[0])}")


22:55:25 | INFO    | Use pytorch device_name: cuda:0
22:55:25 | INFO    | Load pretrained SentenceTransformer: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

22:55:30 | INFO    | Tempo de processamento (all-MiniLM-L6-v2): 0.12258744239807129 segundos
22:55:30 | INFO    |   - Dimensões do vetor (all-MiniLM-L6-v2): 384


In [ ]:
# BGE-Large

bge_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

start_time = time.time()
embeddings_bge = bge_embeddings.embed_documents(textos_teste)
end_time = time.time()

_tempo_total = end_time - start_time
_tempo_medio = _tempo_total / max(1, len(textos_teste))

meta_bge = {
    "model": "BAAI/bge-large-en-v1.5",
    "n_docs": len(textos_teste),
    "tempo_total_s": _tempo_total,
    "tempo_medio_por_doc_s": _tempo_medio,
    "dimensao_vetor": len(embeddings_bge[0]) if embeddings_bge else None
}

meta_summary_bge = (
    f"model={meta_bge['model']} | n_docs={meta_bge['n_docs']} | "
    f"tempo_total_s={meta_bge['tempo_total_s']:.4f} | tempo_medio_por_doc_s={meta_bge['tempo_medio_por_doc_s']:.4f} | "
    f"dimensao_vetor={meta_bge['dimensao_vetor']}"
)

logger.info(f"Tempo de processamento (BAAI/bge-large-en-v1.5): {end_time - start_time} segundos")
logger.info(f"  - Dimensões do vetor (BAAI/bge-large-en-v1.5): {len(embeddings_bge[0])}")


22:55:30 | INFO    | Use pytorch device_name: cuda:0
22:55:30 | INFO    | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

22:55:53 | INFO    | Tempo de processamento (BAAI/bge-large-en-v1.5): 0.10692954063415527 segundos
22:55:53 | INFO    |   - Dimensões do vetor (BAAI/bge-large-en-v1.5): 1024


In [ ]:
import plotly.express as px

# dados (usa as variáveis meta_* que você já criou)
_models = [meta, meta_multilingual_e5, meta_minilm, meta_bge]

# gráfico: dimensão_do_vetor (x) vs tempo_total (y), tamanho = n_docs
fig = px.scatter(
    _models,
    x="dimensao_vetor",
    y="tempo_total_s",
    size="n_docs",
    hover_name="model",
    text="model",
    title="Tempo de processamento vs Dimensão do vetor (embeddings)",
    labels={
        "dimensao_vetor": "Dimensão do vetor",
        "tempo_total_s": "Tempo total (s)",
        "n_docs": "Número de documentos"
    }
)

fig.update_traces(textposition="top center")
fig.update_layout(legend_title_text="n_docs (tamanho do ponto)")
fig.show()

In [ ]:
!pip install -q umap-learn scikit-learn plotly --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-pinecone 0.2.13 requires numpy!=2.0.2,>=1.26.4, but you have numpy 2.0.2 which is incompatible.


In [ ]:
# -------------------------
# Notebook: extração, cache e métricas para comparação de embeddings
# -------------------------
import os
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances # Importar pairwise_distances
import umap
import plotly.express as px
import plotly.graph_objects as go
from scipy.linalg import orthogonal_procrustes
import logging

# Configure logging globally if not already done
if not logging.getLogger().handlers:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# -------------------------
# Ajuste aqui: seus clientes de embeddings (LangChain-like)
# -------------------------
# Exemplo (já informados por você)
# gemini_embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
# multilingual_e5_large_embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")
# minilm_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# bge_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")

models_clients = [
    ("Gemini", gemini_embeddings),
    ("Multilingual-e5-large", multilingual_e5_large_embeddings),
    ("MiniLM", minilm_embeddings),
    ("BGE-large", bge_embeddings),
]

# Textos-alvo (mesma ordem para todos os modelos)
textos_teste = [
    "Qual é a política de férias da nossa empresa?",
    "Preciso de um relatório de despesas de viagem.",
    "Como configuro o acesso à rede privada virtual (VPN)?",
    "Onde encontro o código de conduta da organização?",
    "Quero entender o processo de avaliação de performance."
]

# -------------------------
# Paths / cache
# -------------------------
CACHE_DIR = "embeddings_cache"
METRICS_DIR = "metrics_output"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

# -------------------------
# Função para extrair embeddings com cache em .npy
# -------------------------
def get_embeddings_for_model(model_name, emb_client, texts, batch_size=32, force_recompute=False, sleep_between_batches=0.0):
    """
    Retorna np.array(shape=(n_texts, dim)) e salva/usa cache em embeddings_cache/<model_name>.npy
    - emb_client: objeto com embed_documents(list[str]) ou embed_query(str)
    - force_recompute: ignora cache e recalcula
    """
    cache_path = os.path.join(CACHE_DIR, f"{model_name}.npy")
    if os.path.exists(cache_path) and not force_recompute:
        logging.info(f"Carregando cache {cache_path}")
        arr = np.load(cache_path)
        # Se cache tiver mais itens do que textos pedimos, truncamos para alinhamento
        if arr.shape[0] >= len(texts):
            return arr[:len(texts)]
        # caso contrário, recalcule (incompleto)
        logging.info(f"Cache {cache_path} tem {arr.shape[0]} itens (esperado {len(texts)}). Recomputando.")

    # Se emb_client já for array-like (ex: embeddings_multilingual_e5_large)
    if isinstance(emb_client, (list, np.ndarray)):
        arr = np.array(emb_client)
        np.save(cache_path, arr)
        logging.info(f"Salvo cache (pre-computed) {cache_path} shape={arr.shape}")
        return arr[:len(texts)]


    # Se tiver embed_documents (comum em LangChain)
    if hasattr(emb_client, "embed_documents"):
        batches = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            try:
                emb_batch = emb_client.embed_documents(batch)
            except TypeError:
                # fallback para métodos com assinatura diferente
                emb_batch = emb_client.embed(batch) # Algumas classes usam .embed()
            emb_batch = np.array(emb_batch)
            batches.append(emb_batch)
            if sleep_between_batches:
                time.sleep(sleep_between_batches)
        arr = np.vstack(batches)
        np.save(cache_path, arr)
        logging.info(f"Salvo cache {cache_path} shape={arr.shape}")
        return arr[:len(texts)]

    # Se tiver embed_query (um por vez)
    if hasattr(emb_client, "embed_query"):
        embs = []
        for t in texts:
            e = emb_client.embed_query(t)
            embs.append(np.array(e))
        arr = np.vstack(embs)
        np.save(cache_path, arr)
        logging.info(f"Salvo cache {cache_path} shape={arr.shape}")
        return arr[:len(texts)]

    raise RuntimeError(f"Não sei extrair embeddings de {type(emb_client)}; adapte a função get_embeddings_for_model.")


# -------------------------
# Extração (or loading) de embeddings for all models
# -------------------------
models_arrays = []
model_names = []
print("Extraindo / carregando embeddings para cada modelo:")
# Adicione os arrays de embeddings pré-computados aqui também
precomputed_embeddings = {
    "Gemini": embeddings_gemini,
    "Multilingual-e5-large": embeddings_multilingual_e5_large,
    "MiniLM": embeddings_minilm,
    "BGE-large": embeddings_bge,
}

for name, client in models_clients:
    start = time.time()
    # Try to get pre-computed embeddings first
    if name in precomputed_embeddings and precomputed_embeddings[name] is not None:
        arr = np.array(precomputed_embeddings[name])
        logging.info(f"Usando embeddings pré-computados para {name}. Shape={arr.shape}")
        arr = arr[:len(textos_teste)] # Ensure correct size
    else:
        # Fallback to computing/loading from cache
        arr = get_embeddings_for_model(name, client, textos_teste, batch_size=16, force_recompute=False)

    elapsed = time.time() - start
    logging.info(f"{name}: embeddings shape = {arr.shape}  (tempo {elapsed:.2f}s)")
    if arr.shape[0] < len(textos_teste):
        raise ValueError(f"{name} retornou menos embeddings ({arr.shape[0]}) que textos ({len(textos_teste)})")
    models_arrays.append(arr[:len(textos_teste)])
    model_names.append(name)

# Salva também um arquivo com meta info
# Corrigindo a criação do dicionário para o DataFrame
meta = {"models": model_names, "n_texts": [len(textos_teste)]} # Envolvendo n_texts em uma lista
pd.DataFrame.from_dict(meta, orient='index').to_csv(os.path.join(METRICS_DIR, "meta_models.csv"))

# -------------------------
# Verifica dimensões e decide estratégia
# -------------------------
dims = [arr.shape[1] for arr in models_arrays]
logging.info(f"Dimensões por modelo: {dict(zip(model_names, dims))}")
same_dim = all(d == dims[0] for d in dims)
n_models = len(models_arrays)
n_docs = len(textos_teste)

# -------------------------
# Geração de componentes 3D e alinhamento
# -------------------------
aligned_list = []

if same_dim:
    # PCA conjunta (comparável entre modelos)
    logging.info("Todas as dimensões são iguais -> PCA conjunta")
    X_all = np.vstack(models_arrays)  # shape = (n_models * n_docs, dim)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_all)
    pca = PCA(n_components=3, random_state=RANDOM_STATE)
    X_pca3 = pca.fit_transform(X_scaled)
    # dividimos em blocos por modelo
    for i in range(n_models):
        start = i * n_docs
        end = start + n_docs
        aligned_list.append(X_pca3[start:end, :])
    # guardamos explained variance
    explained_variance = pca.explained_variance_ratio_
    logging.info(f"Explained variance (PC1..3): {explained_variance.round(4)} (soma {explained_variance.sum().round(4)})")
    # save PCA joint for debugging
    np.save(os.path.join(METRICS_DIR,"pca_joint_components.npy"), X_pca3)
else:
    # PCA por-modelo + Procrustes (alinha cada projeção 3D ao primeiro modelo)
    logging.info("Dimensões diferentes -> PCA por-modelo + Procrustes alignment")
    per_model_comps = []
    for i in range(len(models_arrays)): # Iterate using index to access model_names
        arr = models_arrays[i]
        scaler = StandardScaler()
        arr_s = scaler.fit_transform(arr)
        pca = PCA(n_components=3, random_state=RANDOM_STATE)
        comp = pca.fit_transform(arr_s)  # (n_docs, 3)
        per_model_comps.append(comp)
        # opcional: salvar pca por modelo
        np.save(os.path.join(METRICS_DIR, f"pca_{model_names[i]}.npy"), comp)
        logging.info(f"PCA por-modelo {model_names[i]} -> shape {comp.shape}")
    # alinhar com Procrustes: use o primeiro como referência
    ref = per_model_comps[0]
    aligned_list.append(ref)
    for i in range(1, len(per_model_comps)):
        target = per_model_comps[i]
        A = ref - ref.mean(axis=0)
        B = target - target.mean(axis=0)
        R, scale = orthogonal_procrustes(B, A)
        B_aligned = (B @ R) * scale
        B_aligned += ref.mean(axis=0)
        aligned_list.append(B_aligned)
        logging.info(f"Alinhado {model_names[i]} -> referência {model_names[0]}")

# Empilha alinhados para análise conjunta
aligned_3d = np.vstack(aligned_list)  # shape (n_models*n_docs, 3)
np.save(os.path.join(METRICS_DIR, "aligned_components_3d.npy"), aligned_3d)
logging.info(f"aligned_3d shape = {aligned_3d.shape}")

# -------------------------
# Tabela básica para plot / análises
# -------------------------
labels = []
docs = []
for i in range(n_models): # Iterate using index to access model_names
    labels.extend([model_names[i]]*n_docs)
    docs.extend(textos_teste[:n_docs])

# Scores significativos: similarity(doc_emb, centroid) usando embeddings originais (cada modelo)
sizes_scores = []
for i in range(len(models_arrays)): # Iterate using index to access model_names
    arr = models_arrays[i]
    centroid = np.mean(arr, axis=0, keepdims=True)
    # Need to handle potential errors if arr is empty or has invalid shape
    try:
        sims = cosine_similarity(arr, centroid).reshape(-1)
        sizes_scores.extend(sims)
    except ValueError as e:
        logging.warning(f"Could not calculate cosine similarity for model {model_names[i]}: {e}. Appending NaNs.")
        sizes_scores.extend([np.nan] * n_docs) # Append NaNs if calculation fails


sizes_scores = np.array(sizes_scores)

# Mapeamento de tamanho perceptual (compatível NumPy 2.x)
min_sz, max_sz = 6, 30
range_ = np.ptp(sizes_scores)  # np.ptp é compatível com NumPy 2.x
if range_ <= 1e-12 or np.isnan(range_): # Check for NaN range too
    norm = np.zeros_like(sizes_scores)
else:
    norm = (sizes_scores - np.nanmin(sizes_scores)) / range_ # Use nanmin to handle NaNs
sizes_mapped = min_sz + (np.sqrt(norm) * (max_sz - min_sz))
# Replace NaNs in sizes_mapped with a default size (e.g., min_sz) if necessary
sizes_mapped[np.isnan(sizes_mapped)] = min_sz


assert sizes_mapped.shape[0] == len(labels)

df_plot = pd.DataFrame({
    "X": aligned_3d[:,0],
    "Y": aligned_3d[:,1],
    "Z": aligned_3d[:,2],
    "Modelo": labels,
    "Documento": docs,
    "ScoreCentro": sizes_scores,
    "Size": sizes_mapped
})
df_plot["doc_id"] = df_plot.groupby("Modelo").cumcount()

# Salva DF plot (csv)
df_plot.to_csv(os.path.join(METRICS_DIR, "df_plot_aligned.csv"), index=False)

# -------------------------
# Métricas quantitativas: distâncias intra vs inter (usando aligned_3d)
# -------------------------
D_cos = pairwise_distances(aligned_3d, metric='cosine')  # matriz NxN
n_points = D_cos.shape[0]

intra = []
inter = []
models_arr = np.array(df_plot["Modelo"].tolist())
for i in range(n_points):
    for j in range(i+1, n_points):
        if models_arr[i] == models_arr[j]:
            intra.append(D_cos[i,j])
        else:
            inter.append(D_cos[i,j])

intra = np.array(intra)
inter = np.array(inter)

def summarize_array(a):
    # Handle empty or all-NaN arrays
    if a.size == 0 or np.all(np.isnan(a)):
         return {
            "count": int(a.size),
            "mean": np.nan,
            "std": np.nan,
            "median": np.nan,
            "q1": np.nan,
            "q3": np.nan,
            "min": np.nan,
            "max": np.nan
        }
    return {
        "count": int(a.size),
        "mean": float(np.nanmean(a)),
        "std": float(np.nanstd(a)),
        "median": float(np.nanmedian(a)),
        "q1": float(np.nanpercentile(a,25)),
        "q3": float(np.nanpercentile(a,75)),
        "min": float(np.nanmin(a)),
        "max": float(np.nanmax(a))
    }


summary_intra = summarize_array(intra)
summary_inter = summarize_array(inter)

metrics_summary_df = pd.DataFrame([summary_intra, summary_inter], index=["intra_cosine", "inter_cosine"])
metrics_summary_df.to_csv(os.path.join(METRICS_DIR, "distance_summary.csv"))
logging.info("Saved distance summary CSV")

# -------------------------
# Silhouette score (usando aligned_3d, métrica cosine)
# -------------------------
from sklearn.metrics import silhouette_score # Importar silhouette_score

label_ids = pd.Categorical(df_plot["Modelo"]).codes
# silhouette requires at least 2 labels and each label must have >1 sample; here each model has n_docs >=1
# Also needs distinct labels and non-empty data after alignment
try:
    # Check if there are at least 2 unique labels and more than 1 sample
    if len(np.unique(label_ids)) > 1 and len(aligned_3d) > 1:
        sil = silhouette_score(aligned_3d, label_ids, metric='cosine')
    else:
        logging.warning("Cannot compute Silhouette score: less than 2 unique labels or 1 sample.")
        sil = np.nan
except Exception as e:
    logging.warning(f"Silhouette score failed: {e}; definindo NaN")
    sil = np.nan

sil_df = pd.DataFrame({"silhouette_cosine": [float(sil)]})
sil_df.to_csv(os.path.join(METRICS_DIR, "silhouette.csv"), index=False)
logging.info(f"Silhouette (aligned_3d, cosine): {sil}")

# -------------------------
# Centroid distances (matriz entre modelos) usando aligned_3d
# -------------------------
centroids = []
valid_models = []
for name in model_names:
    # Get points for the current model
    model_points = df_plot[df_plot["Modelo"]==name][["X","Y","Z"]].values
    # Only calculate centroid if there are points for the model
    if len(model_points) > 0:
        centroids.append(model_points.mean(axis=0))
        valid_models.append(name)

centroids = np.vstack(centroids) if centroids else np.empty((0, 3)) # Handle case with no valid models

if len(valid_models) > 1: # Need at least 2 centroids to calculate distance matrix
    centroid_dist = pairwise_distances(centroids, metric='cosine')
    centroid_df = pd.DataFrame(centroid_dist, index=valid_models, columns=valid_models)
    centroid_df.to_csv(os.path.join(METRICS_DIR, "centroid_distance_matrix.csv"))
    logging.info("Saved centroid distance matrix")
else:
    logging.warning("Cannot compute centroid distance matrix: less than 2 valid models.")


# -------------------------
# Resumo final de métricas (tabela única)
# -------------------------
final_metrics = {
    "n_models": n_models,
    "n_docs_per_model": n_docs,
    "same_dimensionality": same_dim,
    "silhouette_cosine": float(sil) if not np.isnan(sil) else "NaN" # Handle NaN for CSV
}
# se PCA conjunta, acrescenta explained variance
if same_dim and 'explained_variance' in locals():
    final_metrics["pca_explained_pc1"] = float(explained_variance[0])
    final_metrics["pca_explained_pc1_pc2_pc3_sum"] = float(np.sum(explained_variance))
final_metrics_df = pd.DataFrame.from_dict(final_metrics, orient="index", columns=["value"])
final_metrics_df.to_csv(os.path.join(METRICS_DIR, "final_metrics_summary.csv"))
logging.info("Saved final metrics summary")

# -------------------------
# Plots interativos (Plotly)
# -------------------------
# 3D scatter
if not df_plot.empty: # Only plot if dataframe is not empty
    fig3 = px.scatter_3d(df_plot, x="X", y="Y", z="Z", color="Modelo", size="Size",
                         hover_data=["Documento","Modelo","ScoreCentro"],
                         title="Embeddings 3D (componentes alinhados entre modelos)")
    fig3.update_traces(marker=dict(sizemode="area"))
    fig3.update_layout(margin=dict(l=0,r=0,t=40,b=0))
    fig3.show()
else:
    logging.warning("df_plot is empty. Cannot generate 3D scatter plot.")


# 2D (X vs Y)
if not df_plot.empty: # Only plot if dataframe is not empty
    fig2 = px.scatter(df_plot, x="X", y="Y", color="Modelo", size="Size",
                      hover_data=["Documento","Modelo","ScoreCentro"],
                      title="Projeção 2D (componentes alinhados)")
    fig2.update_traces(marker=dict(sizemode="area"))
    fig2.update_layout(margin=dict(l=0,r=0,t=40,b=0))
    fig2.show()
else:
     logging.warning("df_plot is empty. Cannot generate 2D scatter plot.")


# Histogramas intra vs inter (distâncias cosine)
if len(intra) > 0 or len(inter) > 0: # Only plot if there is data for histograms
    hist = go.Figure()
    if len(intra) > 0:
        hist.add_trace(go.Histogram(x=intra, name='Intra-model (cosine)', opacity=0.75))
    if len(inter) > 0:
        hist.add_trace(go.Histogram(x=inter, name='Inter-model (cosine)', opacity=0.75))
    hist.update_layout(barmode='overlay', title='Histograma Distâncias Cosine: intra vs inter')
    hist.show()
else:
    logging.warning("No data for intra or inter distances. Cannot generate histograms.")


# Also converts histogram statistics to DataFrame and saves
if summary_intra["count"] > 0 or summary_inter["count"] > 0:
    hist_stats_flat = pd.DataFrame([summary_intra, summary_inter], index=["intra_cosine", "inter_cosine"])
    hist_stats_flat.to_csv(os.path.join(METRICS_DIR, "histogram_stats.csv"))
    logging.info("Saved histogram stats CSV")
else:
     logging.warning("No data for histogram stats. Skipping CSV save.")


# -------------------------
# Opcional: UMAP sobre aligned_3d para uma visão alternativa (2D)
# -------------------------
if not df_plot.empty and len(aligned_3d) > 1: # Need data and more than 1 sample for UMAP
    try:
        reducer = umap.UMAP(n_components=2, metric='euclidean', random_state=RANDOM_STATE, min_dist=0.1)
        X_umap2 = reducer.fit_transform(aligned_3d)
        df_plot["UMAP1"] = X_umap2[:,0]
        df_plot["UMAP2"] = X_umap2[:,1]
        df_plot.to_csv(os.path.join(METRICS_DIR, "df_plot_with_umap.csv"), index=False)

        fig_umap = px.scatter(df_plot, x="UMAP1", y="UMAP2", color="Modelo", size="Size",
                             hover_data=["Documento","Modelo","ScoreCentro"],
                             title="UMAP 2D sobre aligned_3d")
        fig_umap.update_traces(marker=dict(sizemode="area"))
        fig_umap.update_layout(margin=dict(l=0,r=0,t=40,b=0))
        fig_umap.show()
        logging.info("Generated and saved UMAP plot and CSV.")
    except Exception as e:
         logging.warning(f"UMAP plotting failed: {e}")
else:
    logging.warning("Cannot generate UMAP plot: df_plot is empty or less than 2 samples.")


# -------------------------
# Save final artifacts
# -------------------------
# Already saved:
# - embeddings cache per model in embeddings_cache/*.npy
# - aligned_3d in metrics_output/aligned_components_3d.npy
# - df_plot csvs, metrics csvs in metrics_output/
logging.info(f"Todos os artefatos salvos em: {os.path.abspath(METRICS_DIR)} e {os.path.abspath(CACHE_DIR)}")

# -------------------------
# Mensagem final
# -------------------------
print("Concluído — embeddings salvos em .npy e métricas geradas (CSV) em:", os.path.abspath(METRICS_DIR))
print("Arquivos de interesse:")
try:
    for fn in sorted(os.listdir(METRICS_DIR)):
        print(" -", fn)
except FileNotFoundError:
    print(f" - Diretório de métricas '{METRICS_DIR}' não encontrado.")

22:57:10 | INFO    | Usando embeddings pré-computados para Gemini. Shape=(5, 3072)
22:57:10 | INFO    | Gemini: embeddings shape = (5, 3072)  (tempo 0.00s)
22:57:10 | INFO    | Usando embeddings pré-computados para Multilingual-e5-large. Shape=(5, 1024)
22:57:10 | INFO    | Multilingual-e5-large: embeddings shape = (5, 1024)  (tempo 0.00s)
22:57:10 | INFO    | Usando embeddings pré-computados para MiniLM. Shape=(5, 384)
22:57:10 | INFO    | MiniLM: embeddings shape = (5, 384)  (tempo 0.00s)
22:57:10 | INFO    | Usando embeddings pré-computados para BGE-large. Shape=(5, 1024)
22:57:10 | INFO    | BGE-large: embeddings shape = (5, 1024)  (tempo 0.00s)
22:57:10 | INFO    | Dimensões por modelo: {'Gemini': 3072, 'Multilingual-e5-large': 1024, 'MiniLM': 384, 'BGE-large': 1024}
22:57:10 | INFO    | Dimensões diferentes -> PCA por-modelo + Procrustes alignment
22:57:10 | INFO    | PCA por-modelo Gemini -> shape (5, 3)
22:57:10 | INFO    | PCA por-modelo Multilingual-e5-large -> shape (5, 3)
2

Extraindo / carregando embeddings para cada modelo:


22:57:11 | INFO    | Saved histogram stats CSV
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



22:57:19 | INFO    | Generated and saved UMAP plot and CSV.
22:57:19 | INFO    | Todos os artefatos salvos em: /content/metrics_output e /content/embeddings_cache


Concluído — embeddings salvos em .npy e métricas geradas (CSV) em: /content/metrics_output
Arquivos de interesse:
 - aligned_components_3d.npy
 - centroid_distance_matrix.csv
 - df_plot_aligned.csv
 - df_plot_with_umap.csv
 - distance_summary.csv
 - final_metrics_summary.csv
 - histogram_stats.csv
 - meta_models.csv
 - pca_BGE-large.npy
 - pca_Gemini.npy
 - pca_MiniLM.npy
 - pca_Multilingual-e5-large.npy
 - silhouette.csv


In [ ]:
import pandas as pd
import plotly.express as px
import os

# Carregar a matriz de distância dos centróides
metrics_dir = "metrics_output"
centroid_matrix_path = os.path.join(metrics_dir, "centroid_distance_matrix.csv")

if os.path.exists(centroid_matrix_path):
    centroid_df = pd.read_csv(centroid_matrix_path, index_col=0)

    # Criar o heatmap
    fig = px.imshow(
        centroid_df,
        text_auto=True, # Mostrar valores no heatmap
        color_continuous_scale="Viridis", # Escala de cor
        title="Heatmap da Matriz de Distância Cosine entre Centróides dos Modelos"
    )

    # Layout para melhor visualização
    fig.update_layout(
        xaxis_title="Modelo",
        yaxis_title="Modelo"
    )

    fig.show()

else:
    print(f"Arquivo {centroid_matrix_path} não encontrado. Execute a célula anterior para gerá-lo.")

In [ ]:
!pip install kaleido

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [kaleido]


In [ ]:
# -------------------------
# métricas extras + salvar plots (HTML/PNG) + linhas por documento
# Código atualizado e com checagens defensivas
# -------------------------
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import pairwise_distances
import plotly.graph_objects as go
import plotly.express as px
import logging
from scipy.linalg import orthogonal_procrustes
import zipfile

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

# Paths
OUTPUT_DIR = METRICS_DIR if 'METRICS_DIR' in globals() else "metrics_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Basic environment checks
required_vars = ["aligned_3d", "df_plot", "models_arrays", "model_names", "n_models", "n_docs"]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(f"Variáveis necessárias ausentes no ambiente: {missing}. Execute a célula anterior antes deste bloco.")

# local refs
aligned_3d = globals()["aligned_3d"]
df_plot = globals()["df_plot"]
models_arrays = globals()["models_arrays"]
model_names = globals()["model_names"]
n_models = int(globals()["n_models"])
n_docs = int(globals()["n_docs"])
textos_teste_local = globals().get("textos_teste", [f"doc_{i}" for i in range(n_docs)])

# --- 1) Per-document cross-model distances ---
per_doc_stats = []
for d in range(n_docs):
    idxs = [m * n_docs + d for m in range(n_models)]
    try:
        pts = aligned_3d[idxs, :]           # shape (n_models, 3)
    except Exception as e:
        logger.error(f"Erro ao indexar aligned_3d para doc {d}: {e}")
        pts = np.empty((0, 3))
    if pts.shape[0] > 1:
        D = pairwise_distances(pts, metric='cosine')
        iu = np.triu_indices(D.shape[0], k=1)
        vals = D[iu]
        vals = vals[~np.isnan(vals)]
        mean_pair = float(np.nanmean(vals)) if vals.size>0 else np.nan
        std_pair = float(np.nanstd(vals)) if vals.size>0 else np.nan
        max_pair = float(np.nanmax(vals)) if vals.size>0 else np.nan
    else:
        mean_pair = np.nan; std_pair=np.nan; max_pair=np.nan
    per_doc_stats.append({
        "doc_id": d,
        "text": textos_teste_local[d] if d < len(textos_teste_local) else f"doc_{d}",
        "mean_cross_model_cosine": mean_pair,
        "std_cross_model_cosine": std_pair,
        "max_cross_model_cosine": max_pair,
        "n_models_compared": int(pts.shape[0])
    })
per_doc_df = pd.DataFrame(per_doc_stats)
per_doc_df.to_csv(os.path.join(OUTPUT_DIR, "per_document_cross_model_distances.csv"), index=False)
logger.info("Saved per-document cross-model distances")

# --- 2) Pairwise distance matrix correlations between models (Spearman) ---
model_distance_flat = {}
for i, name in enumerate(model_names):
    try:
        arr = models_arrays[i][:n_docs]
        D = pairwise_distances(arr, metric='cosine')
    except Exception:
        try:
            block = aligned_3d[i*n_docs:(i+1)*n_docs]
            D = pairwise_distances(block, metric='cosine')
        except Exception as e:
            logger.warning(f"Não foi possível obter matriz de distância para {name}: {e}")
            D = np.full((n_docs, n_docs), np.nan)
    iu = np.triu_indices(n_docs, k=1)
    model_distance_flat[name] = D[iu]

name_list = model_names
m = len(name_list)
spearman_mat = np.full((m,m), np.nan)
pval_mat = np.full((m,m), np.nan)
for i in range(m):
    for j in range(m):
        if i == j:
            spearman_mat[i,j] = 1.0
            pval_mat[i,j] = 0.0
        else:
            a = model_distance_flat[name_list[i]]
            b = model_distance_flat[name_list[j]]
            # remove nan pairs
            mask = (~np.isnan(a)) & (~np.isnan(b))
            if mask.sum() < 2:
                spearman_mat[i,j] = np.nan
                pval_mat[i,j] = np.nan
            else:
                try:
                    r, p = spearmanr(a[mask], b[mask])
                    spearman_mat[i,j] = float(r) if not np.isnan(r) else np.nan
                    pval_mat[i,j] = float(p) if not np.isnan(p) else np.nan
                except Exception as e:
                    logger.warning(f"Spearman failed for {name_list[i]} vs {name_list[j]}: {e}")
                    spearman_mat[i,j] = np.nan
                    pval_mat[i,j] = np.nan

spearman_df = pd.DataFrame(spearman_mat, index=name_list, columns=name_list)
pval_df = pd.DataFrame(pval_mat, index=name_list, columns=name_list)
spearman_df.to_csv(os.path.join(OUTPUT_DIR, "distance_matrix_spearman_corr.csv"))
pval_df.to_csv(os.path.join(OUTPUT_DIR, "distance_matrix_spearman_pvals.csv"))
logger.info("Saved Spearman correlation between distance matrices")

# --- 3) Procrustes residuals (if PCA per-model files exist) ---
procrustes_res = []
pca_files = {}
for name in model_names:
    path = os.path.join(OUTPUT_DIR, f"pca_{name}.npy")
    if os.path.exists(path):
        try:
            pca_files[name] = np.load(path)
        except Exception as e:
            logger.warning(f"Could not load {path}: {e}")

if model_names[0] in pca_files:
    ref_name = model_names[0]
    ref = pca_files[ref_name]
    A = ref - ref.mean(axis=0)
    for name in model_names[1:]:
        if name in pca_files:
            tgt = pca_files[name]
            B = tgt - tgt.mean(axis=0)
            try:
                R, scale = orthogonal_procrustes(B, A)
                B_aligned = (B @ R) * scale + ref.mean(axis=0)
                # compute Frobenius norm of difference between A and (B_aligned - ref.mean)
                resid = np.linalg.norm(A - (B_aligned - ref.mean(axis=0)), ord='fro')
                procrustes_res.append({"model": name, "residual_norm": float(resid)})
            except Exception as e:
                procrustes_res.append({"model": name, "residual_norm": np.nan, "error": str(e)})
procrustes_df = pd.DataFrame(procrustes_res)
procrustes_df.to_csv(os.path.join(OUTPUT_DIR, "procrustes_residuals.csv"), index=False)
logger.info("Saved Procrustes residuals (if computed)")

# --- 4) Average displacement per model vs reference (euclidean in aligned_3d) ---
disp_stats = []
ref_block = aligned_3d[0*n_docs:(0+1)*n_docs] if aligned_3d.shape[0] >= n_docs else np.empty((0,3))
for i, name in enumerate(model_names):
    block = aligned_3d[i*n_docs:(i+1)*n_docs]
    if block.shape[0] != ref_block.shape[0]:
        logger.warning(f"Block size mismatch for model {name}: {block.shape} vs ref {ref_block.shape}")
    # compute distances safely
    try:
        dists = np.linalg.norm(block - ref_block, axis=1)
        mean_d = float(np.nanmean(dists))
        std_d = float(np.nanstd(dists))
    except Exception:
        mean_d = np.nan
        std_d = np.nan
    disp_stats.append({
        "model": name,
        "mean_euclidean_to_ref": mean_d,
        "std_euclidean_to_ref": std_d
    })
disp_df = pd.DataFrame(disp_stats)
disp_df.to_csv(os.path.join(OUTPUT_DIR, "model_to_ref_displacement.csv"), index=False)
logger.info("Saved model displacement stats")

# --- 5) Augment 3D plot: draw lines connecting same doc across models (trajectories) ---
if df_plot is None or df_plot.empty:
    logger.warning("df_plot vazio - pulando geração de plot 3D.")
else:
    fig3 = px.scatter_3d(df_plot, x="X", y="Y", z="Z", color="Modelo", size="Size",
                         hover_data=["Documento","Modelo","ScoreCentro"],
                         title="Embeddings 3D (componentes alinhados) + doc-trajectories")
    fig3.update_traces(marker=dict(sizemode="area"))
    for d in range(n_docs):
        xs = [aligned_3d[m*n_docs + d, 0] for m in range(n_models)]
        ys = [aligned_3d[m*n_docs + d, 1] for m in range(n_models)]
        zs = [aligned_3d[m*n_docs + d, 2] for m in range(n_models)]
        fig3.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                                    line=dict(color='gray', width=2), showlegend=False, hoverinfo='none'))
    html_path = os.path.join(OUTPUT_DIR, "embeddings_3d_trajectories.html")
    png_path = os.path.join(OUTPUT_DIR, "embeddings_3d_trajectories.png")
    fig3.write_html(html_path)
    # try to save png using kaleido (preferred)
    try:
        fig3.write_image(png_path, engine="kaleido")
        logger.info(f"Saved 3D plot as PNG: {png_path}")
    except Exception as e:
        logger.warning(f"Could not save PNG (3D) - HTML saved. Error: {e}")

# --- 6) Save UMAP + PCA 2D as HTML and PNG (if exists) ---
if "UMAP1" in df_plot.columns and "UMAP2" in df_plot.columns:
    fig_u = px.scatter(df_plot, x="UMAP1", y="UMAP2", color="Modelo", size="Size",
                       hover_data=["Documento","Modelo","ScoreCentro"],
                       title="UMAP 2D sobre aligned_3d")
    html_umap = os.path.join(OUTPUT_DIR, "umap_2d.html")
    png_umap = os.path.join(OUTPUT_DIR, "umap_2d.png")
    fig_u.write_html(html_umap)
    try:
        fig_u.write_image(png_umap, engine="kaleido")
        logger.info(f"Saved UMAP PNG: {png_umap}")
    except Exception as e:
        logger.warning(f"Could not save PNG (UMAP) - HTML saved. Error: {e}")
else:
    logger.info("UMAP columns not found in df_plot - skipping UMAP png/html save.")

# --- 7) Save Spearman heatmap HTML (and PNG if possible) ---
spearman_path = os.path.join(OUTPUT_DIR, "distance_matrix_spearman_corr.csv")
if os.path.exists(spearman_path):
    try:
        spearman_df = pd.read_csv(spearman_path, index_col=0)
        fig_hm = px.imshow(spearman_df, text_auto=True, title="Spearman corr. entre matrizes de distância (modelos)")
        html_hm = os.path.join(OUTPUT_DIR, "spearman_heatmap.html")
        png_hm = os.path.join(OUTPUT_DIR, "spearman_heatmap.png")
        fig_hm.write_html(html_hm)
        try:
            fig_hm.write_image(png_hm, engine="kaleido")
            logger.info(f"Saved Spearman heatmap PNG: {png_hm}")
        except Exception as e:
            logger.warning(f"Could not save PNG (heatmap) - HTML saved. Error: {e}")
    except Exception as e:
        logger.warning(f"Could not create heatmap from {spearman_path}: {e}")
else:
    logger.info("Spearman CSV not found - skipping heatmap.")

# --- 8) Boxplot of per-doc mean cross-model distances (HTML/PNG) ---
per_doc_csv = os.path.join(OUTPUT_DIR, "per_document_cross_model_distances.csv")
if os.path.exists(per_doc_csv):
    pdf = pd.read_csv(per_doc_csv)
    fig_box = px.box(pdf, y="mean_cross_model_cosine", points="all",
                     title="Distribuição das distâncias cross-model por documento (média)")
    html_box = os.path.join(OUTPUT_DIR, "per_doc_cross_model_box.html")
    png_box = os.path.join(OUTPUT_DIR, "per_doc_cross_model_box.png")
    fig_box.write_html(html_box)
    try:
        fig_box.write_image(png_box, engine="kaleido")
        logger.info(f"Saved per-doc boxplot PNG: {png_box}")
    except Exception as e:
        logger.warning(f"Could not save PNG (boxplot) - HTML saved. Error: {e}")
else:
    logger.info("Per-document CSV not found - skipping boxplot.")

# --- 9) Zip artifacts (CSV/NPY/HTML/PNG) ---
zip_path = os.path.join(OUTPUT_DIR, "metrics_bundle.zip")
with zipfile.ZipFile(zip_path, 'w') as zf:
    for fn in sorted(os.listdir(OUTPUT_DIR)):
        if fn == os.path.basename(zip_path):
            continue
        if fn.endswith(('.csv', '.npy', '.html', '.png')):
            zf.write(os.path.join(OUTPUT_DIR, fn), arcname=fn)
logger.info(f"Zipped metrics bundle -> {zip_path}")

# --- 10) Diagnostics text report ---
diag_lines = []
sil_path = os.path.join(OUTPUT_DIR, "silhouette.csv")
sil_val = None
if os.path.exists(sil_path):
    try:
        sil_val = pd.read_csv(sil_path).iloc[0,0]
    except Exception:
        sil_val = None
diag_lines.append(f"Silhouette (see silhouette.csv): {sil_val}")
diag_lines.append("Mean cross-model distance per doc: per_document_cross_model_distances.csv")
diag_lines.append("Spearman matrix: distance_matrix_spearman_corr.csv")
diag_lines.append("")
diag_lines.append("Top per-doc mean cross-model (lowest = most consistent across models):")
try:
    diag_lines.append(per_doc_df.sort_values("mean_cross_model_cosine").head().to_string(index=False))
except Exception:
    diag_lines.append("Could not stringify per_doc_df")
with open(os.path.join(OUTPUT_DIR, "diagnostics_report.txt"), "w") as fh:
    fh.write("\n".join(diag_lines))
logger.info("Saved diagnostics_report.txt")

print("Métricas extras calculadas e arquivos salvos em:", os.path.abspath(OUTPUT_DIR))
print("Arquivos:", sorted(os.listdir(OUTPUT_DIR)))


# Re-salvar plots como HTML + PNG (robusto para Colab / kaleido)
import importlib, sys, os, logging, subprocess
import numpy as np, pandas as pd
import plotly.express as px, plotly.graph_objects as go
from sklearn.metrics.pairwise import pairwise_distances

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
OUTPUT_DIR = METRICS_DIR if 'METRICS_DIR' in globals() else "metrics_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 0) instalar/importar kaleido se necessário
def ensure_kaleido():
    try:
        import kaleido
        logger.info(f"kaleido importado: {kaleido.__version__}")
        return True
    except Exception:
        logger.info("kaleido não encontrado -> instalando via pip...")
        try:
            # instala no runtime atual
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "kaleido"])
            import importlib
            import kaleido
            logger.info(f"kaleido instalado: {kaleido.__version__}")
            return True
        except Exception as e:
            logger.warning(f"Falha ao instalar/importar kaleido: {e}")
            return False

kaleido_available = ensure_kaleido()

# 1) reload plotly IO para tentar vincular engine sem reiniciar runtime
if kaleido_available:
    try:
        import plotly
        import plotly.io as pio
        importlib.reload(plotly)
        importlib.reload(pio)
        logger.info("Plotly e plotly.io recarregados.")
    except Exception as e:
        logger.warning(f"Falha ao recarregar plotly modules: {e}")

# small helper to attempt save HTML + PNG
def save_fig(fig, basename, width=1200, height=800, scale=2):
    html_path = os.path.join(OUTPUT_DIR, f"{basename}.html")
    png_path = os.path.join(OUTPUT_DIR, f"{basename}.png")
    try:
        fig.write_html(html_path)
    except Exception as e:
        logger.warning(f"Falha ao salvar HTML {html_path}: {e}")
    if kaleido_available:
        try:
            # engine specified explicitly
            fig.write_image(png_path, engine="kaleido", width=width, height=height, scale=scale)
            logger.info(f"Saved PNG: {png_path}")
        except Exception as e:
            logger.warning(f"Could not save PNG ({basename}): {e}. HTML saved at {html_path}")
    else:
        logger.warning(f"kaleido não disponível — salvo somente HTML: {html_path}")
    return html_path, (png_path if os.path.exists(png_path) else None)

# Validate environment variables
required = ["df_plot", "aligned_3d", "model_names", "n_docs", "n_models"]
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(f"Variáveis ausentes no ambiente para recriar plots: {missing}. Execute a célula anterior primeiro.")

df_plot = globals()["df_plot"]
aligned_3d = globals()["aligned_3d"]
model_names = globals()["model_names"]
n_docs = int(globals()["n_docs"])
n_models = int(globals()["n_models"])

# 2) Re-criar fig3 (3D com trajetórias)
try:
    fig3 = px.scatter_3d(df_plot, x="X", y="Y", z="Z", color="Modelo", size="Size",
                         hover_data=["Documento","Modelo","ScoreCentro"],
                         title="Embeddings 3D (componentes alinhados) + doc-trajectories")
    fig3.update_traces(marker=dict(sizemode="area"))
    # linhas conectando pontos do mesmo documento (ordem de modelos)
    for d in range(n_docs):
        xs = [aligned_3d[m*n_docs + d, 0] for m in range(n_models)]
        ys = [aligned_3d[m*n_docs + d, 1] for m in range(n_models)]
        zs = [aligned_3d[m*n_docs + d, 2] for m in range(n_models)]
        fig3.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                                    line=dict(color='gray', width=2), showlegend=False, hoverinfo='none'))
    save_fig(fig3, "embeddings_3d_trajectories")
except Exception as e:
    logger.error(f"Erro ao (re)criar/salvar fig3: {e}")

# 3) Re-criar fig_umap (se colunas UMAP existirem)
if "UMAP1" in df_plot.columns and "UMAP2" in df_plot.columns:
    try:
        fig_u = px.scatter(df_plot, x="UMAP1", y="UMAP2", color="Modelo", size="Size",
                           hover_data=["Documento","Modelo","ScoreCentro"], title="UMAP 2D")
        save_fig(fig_u, "umap_2d")
    except Exception as e:
        logger.error(f"Erro ao (re)criar/salvar UMAP: {e}")
else:
    logger.info("Colunas UMAP não encontradas em df_plot — pulando UMAP.")

# 4) Re-criar heatmap Spearman (se arquivo existente)
spearman_csv = os.path.join(OUTPUT_DIR, "distance_matrix_spearman_corr.csv")
if os.path.exists(spearman_csv):
    try:
        spearman_df = pd.read_csv(spearman_csv, index_col=0)
        fig_hm = px.imshow(spearman_df, text_auto=True, title="Spearman corr. entre matrizes de distância (modelos)")
        save_fig(fig_hm, "spearman_heatmap")
    except Exception as e:
        logger.error(f"Erro ao (re)criar/salvar heatmap Spearman: {e}")
else:
    logger.info("Spearman CSV não encontrado — pulando heatmap.")

# 5) Re-criar boxplot per-doc cross-model (se csv available)
per_doc_csv = os.path.join(OUTPUT_DIR, "per_document_cross_model_distances.csv")
if os.path.exists(per_doc_csv):
    try:
        pdf = pd.read_csv(per_doc_csv)
        fig_box = px.box(pdf, y="mean_cross_model_cosine", points="all",
                         title="Distribuição das distâncias cross-model por documento (média)")
        save_fig(fig_box, "per_doc_cross_model_box")
    except Exception as e:
        logger.error(f"Erro ao (re)criar/salvar boxplot: {e}")
else:
    logger.info("Per-document CSV não encontrado — pulando boxplot.")

# 6) listar arquivos no OUTPUT_DIR
print("Arquivos no diretório de métricas (após tentativa de salvar PNGs):")
for fn in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", fn)

# 7) se ainda houver problema com PNGs: instrução final
if not kaleido_available:
    print("\nNota: kaleido não está disponível no runtime. Execute:\n  !pip install -U kaleido\n e re-execute esta célula. Se ainda falhar, reinicie o runtime e execute novamente.")
else:
    print("\nSe PNGs não foram gerados apesar do kaleido estar instalado, reinicie o runtime do Colab e reexecute as células (às vezes necessário).")



22:57:27 | INFO    | Saved per-document cross-model distances
22:57:27 | INFO    | Saved Spearman correlation between distance matrices
22:57:27 | INFO    | Saved Procrustes residuals (if computed)
22:57:27 | INFO    | Saved model displacement stats
22:57:27 | WARNING | Could not save PNG (3D) - HTML saved. Error: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

22:57:27 | WARNING | Could not save PNG (UMAP) - HTML saved. Error: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

22:57:27 | WARNING | Could not save PNG (heatmap) - HTML saved. Error: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido

22:57:27 | WARNING | Could not save PNG (boxplot) - HTML saved. Error: 
Image export using the "kaleido" engine requires the kaleido package

Métricas extras calculadas e arquivos salvos em: /content/metrics_output
Arquivos: ['aligned_components_3d.npy', 'centroid_distance_matrix.csv', 'df_plot_aligned.csv', 'df_plot_with_umap.csv', 'diagnostics_report.txt', 'distance_matrix_spearman_corr.csv', 'distance_matrix_spearman_pvals.csv', 'distance_summary.csv', 'embeddings_3d_trajectories.html', 'final_metrics_summary.csv', 'histogram_stats.csv', 'meta_models.csv', 'metrics_bundle.zip', 'model_to_ref_displacement.csv', 'pca_BGE-large.npy', 'pca_Gemini.npy', 'pca_MiniLM.npy', 'pca_Multilingual-e5-large.npy', 'per_doc_cross_model_box.html', 'per_document_cross_model_distances.csv', 'procrustes_residuals.csv', 'silhouette.csv', 'spearman_heatmap.html', 'umap_2d.html']


22:57:32 | WARNING | Falha ao instalar/importar kaleido: module 'kaleido' has no attribute '__version__'
22:57:32 | WARNING | kaleido não disponível — salvo somente HTML: metrics_output/embeddings_3d_trajectories.html
22:57:32 | WARNING | kaleido não disponível — salvo somente HTML: metrics_output/umap_2d.html
22:57:32 | WARNING | kaleido não disponível — salvo somente HTML: metrics_output/spearman_heatmap.html
22:57:32 | WARNING | kaleido não disponível — salvo somente HTML: metrics_output/per_doc_cross_model_box.html


Arquivos no diretório de métricas (após tentativa de salvar PNGs):
 - aligned_components_3d.npy
 - centroid_distance_matrix.csv
 - df_plot_aligned.csv
 - df_plot_with_umap.csv
 - diagnostics_report.txt
 - distance_matrix_spearman_corr.csv
 - distance_matrix_spearman_pvals.csv
 - distance_summary.csv
 - embeddings_3d_trajectories.html
 - final_metrics_summary.csv
 - histogram_stats.csv
 - meta_models.csv
 - metrics_bundle.zip
 - model_to_ref_displacement.csv
 - pca_BGE-large.npy
 - pca_Gemini.npy
 - pca_MiniLM.npy
 - pca_Multilingual-e5-large.npy
 - per_doc_cross_model_box.html
 - per_document_cross_model_distances.csv
 - procrustes_residuals.csv
 - silhouette.csv
 - spearman_heatmap.html
 - umap_2d.html

Nota: kaleido não está disponível no runtime. Execute:
  !pip install -U kaleido
 e re-execute esta célula. Se ainda falhar, reinicie o runtime e execute novamente.


###Análise de Qualidade Semântica

In [ ]:
pergunta = "Quero tirar uns dias de folga do trabalho."

emb_pergunta_gemini = gemini_embeddings.embed_query(pergunta)
emb_pergunta_multilingual_e5_large = multilingual_e5_large_embeddings.embed_query(pergunta)
emb_pergunta_minilm = minilm_embeddings.embed_query(pergunta)
emb_pergunta_bge = bge_embeddings.embed_query(pergunta)

In [ ]:
modelos = {
    "Gemini": (emb_pergunta_gemini, embeddings_gemini),
    "Multilingual-e5-large": (emb_pergunta_multilingual_e5_large,
                              embeddings_multilingual_e5_large),
    "MiniLM": (emb_pergunta_minilm, embeddings_minilm),
    "BGE-large": (emb_pergunta_bge, embeddings_bge)
}

print(pergunta)

Quero tirar uns dias de folga do trabalho.


In [ ]:
for nome, (emb_q, emb_docs) in modelos.items():
  similaridades = cosine_similarity([emb_q], emb_docs)[0]
  doc_e_similaridade = sorted(
      zip(textos_teste, similaridades), key=lambda x: x[1], reverse=True
  )
  print(f"--- Ranking para o modelo {nome} ---")
  for i, (doc, sim) in enumerate(doc_e_similaridade[:3], 1):
    print(f"  {i}. (Score: {sim:.3f}) {doc}")
  print()

--- Ranking para o modelo Gemini ---
  1. (Score: 0.736) Qual é a política de férias da nossa empresa?
  2. (Score: 0.675) Preciso de um relatório de despesas de viagem.
  3. (Score: 0.649) Quero entender o processo de avaliação de performance.

--- Ranking para o modelo Multilingual-e5-large ---
  1. (Score: 0.856) Qual é a política de férias da nossa empresa?
  2. (Score: 0.848) Preciso de um relatório de despesas de viagem.
  3. (Score: 0.839) Quero entender o processo de avaliação de performance.

--- Ranking para o modelo MiniLM ---
  1. (Score: 0.496) Quero entender o processo de avaliação de performance.
  2. (Score: 0.469) Onde encontro o código de conduta da organização?
  3. (Score: 0.465) Qual é a política de férias da nossa empresa?

--- Ranking para o modelo BGE-large ---
  1. (Score: 0.646) Qual é a política de férias da nossa empresa?
  2. (Score: 0.645) Preciso de um relatório de despesas de viagem.
  3. (Score: 0.621) Onde encontro o código de conduta da organização?



---

### 📊 Observações sobre o teste de embeddings com múltiplas queries

O teste utilizou várias queries de exemplo, como consultas sobre folga, VPN, relatórios e código de conduta, avaliando quatro modelos de embeddings: **Gemini**, **MiniLM (all-MiniLM-L6-v2)**, **Multilingual-e5-large** e **BGE-large**.

---

### Principais resultados qualitativos

* **Gemini (embedding-001)**
  * Em queries curtas e focadas, às vezes prioriza a estrutura da frase ao invés do conceito central.
  * Mostra boa capacidade de captura semântica, mas scores podem variar dependendo do tema.

* **MiniLM (all-MiniLM-L6-v2)**
  * Rápido e leve, adequado para cenários de baixa latência.
  * Perde precisão semântica em queries mais específicas ou em idiomas fora do inglês predominante do treinamento.

* **Multilingual-e5-large**
  * Robusto para múltiplos idiomas, inclusive português.
  * Captura bem relações semânticas e traz respostas relevantes no Top-1 com maior consistência de score.

* **BGE-large (BAAI/bge-large-en-v1.5)**
  * Embeddings ajustados para retrieval, conseguem priorizar corretamente o conceito central da query.
  * Mesmo que scores absolutos possam ser menores que outros modelos, a relevância do Top-1 geralmente é alta.

---

### Por que isso acontece?

* **Treinamento e foco linguístico**
  * BGE-large foi otimizado para retrieval e captura nuances semânticas de forma consistente.
  * Gemini ainda está em pré-lançamento, podendo favorecer padrões de frase mais do que o conceito.
  * MiniLM é projetado para velocidade, com embeddings mais genéricos.
  * Multilingual-e5-large é sólido para consultas multilíngues, incluindo português.

* **Capacidade e dimensão do modelo**
  * Modelos maiores, como BGE-large (≈1B parâmetros) e Gemini (3072 dimensões), carregam mais nuances semânticas.
  * Modelos menores ou mais leves, como MiniLM (33M parâmetros, 384 dimensões), são rápidos, mas menos precisos.

* **Dados de treinamento**
  * Nem todos os modelos foram igualmente expostos a português ou a domínios corporativos.
  * BGE-large e Multilingual-e5-large tendem a lidar melhor com queries multilíngues.

---

### ⚠️ Considerações

* Um único teste não permite conclusões estatísticas definitivas.
* Para análise robusta, recomenda-se rodar **várias queries reais**, computando métricas como **Top-k accuracy** e **Mean Reciprocal Rank (MRR)**.
* Scores absolutos não são comparáveis entre modelos; a análise deve focar na **ordem relativa do ranking** para cada modelo.
* Em cenários críticos, considere **rerankers** ou validação humana para garantir a relevância das respostas.

---

### ✅ Recomendações práticas

1. Continue testando com queries representativas do seu domínio.
2. Avalie **Top-k** e **MRR** para cada modelo.
3. Considere trade-offs:
   * **BGE-large**: ótima precisão e relevância, ideal offline.
   * **Multilingual-e5-large**: alta consistência em múltiplos idiomas.
   * **Gemini**: modelo em evolução, potencial de melhoria rápida.
   * **MiniLM**: velocidade e baixo custo, mas menor precisão semântica.


---

### 📊 Desempenho dos modelos de embeddings

#### **BGE-large**
- **Top-1**: ✅
- **Top-3**: ✅
- **Top-5**: ✅
- **MRR**: 1.0
- **Observações**: Captura bem o conceito central; embeddings ajustados para retrieval.

---

#### **Gemini**
- **Top-1**: ✅
- **Top-3**: ✅
- **Top-5**: ✅
- **MRR**: 1.0
- **Observações**: Às vezes prioriza a estrutura da frase, mas já mostra boa semântica.

---

#### **Multilingual-e5-large**
- **Top-1**: ✅
- **Top-3**: ✅
- **Top-5**: ✅
- **MRR**: 1.0
- **Observações**: Robustez em múltiplos idiomas, inclusive português.

---

#### **MiniLM**
- **Top-1**: ❌
- **Top-3**: ✅
- **Top-5**: ✅
- **MRR**: 0.17
- **Observações**: Rápido e leve, mas perde precisão em queries curtas ou específicas.



---

### 🔍 Explicando as métricas

* **Top-1**: Indica se o documento mais relevante foi retornado na primeira posição.  
  ✅ = correto, ❌ = incorreto  

* **Top-3**: Mostra se o documento relevante está entre os três primeiros retornos.  

* **Top-5**: Verifica se o documento relevante aparece entre os cinco primeiros.  

* **MRR (Mean Reciprocal Rank)**: Métrica que avalia a posição do documento relevante; quanto mais próximo do topo, maior o valor (máximo = 1.0).

---

### ⚖️ Considerações pedagógicas

* **Treinamento e foco linguístico**: modelos grandes ou multilíngues capturam melhor nuances semânticas.  
* **Dimensão do vetor**: embeddings maiores tendem a guardar mais informação semântica, mas exigem mais memória.  
* **Dados de treino**: exposição a português ou termos corporativos afeta diretamente o ranking.  

---


In [ ]:
import pandas as pd

# Dados comparativos
data = [
    {
        "Modelo": "Gemini",
        "Top-1 retornado": "Qual é a política de férias da nossa empresa?",
        "Score Top-1": 0.831,
        "Acertou?": True
    },
    {
        "Modelo": "Multilingual-e5-large",
        "Top-1 retornado": "Qual é a política de férias da nossa empresa?",
        "Score Top-1": 0.856,
        "Acertou?": True
    },
    {
        "Modelo": "MiniLM",
        "Top-1 retornado": "Quero entender o processo de avaliação de performance.",
        "Score Top-1": 0.496,
        "Acertou?": False
    },
    {
        "Modelo": "BGE-large",
        "Top-1 retornado": "Qual é a política de férias da nossa empresa?",
        "Score Top-1": 0.646,
        "Acertou?": True
    }
]

df = pd.DataFrame(data)
print(df)


                  Modelo                                    Top-1 retornado  \
0                 Gemini      Qual é a política de férias da nossa empresa?   
1  Multilingual-e5-large      Qual é a política de férias da nossa empresa?   
2                 MiniLM  Quero entender o processo de avaliação de perf...   
3              BGE-large      Qual é a política de férias da nossa empresa?   

   Score Top-1  Acertou?  
0        0.831      True  
1        0.856      True  
2        0.496     False  
3        0.646      True  


In [ ]:
import plotly.express as px

fig = px.bar(
    df,
    x="Modelo",
    y="Score Top-1",
    color="Acertou?",
    text="Score Top-1",
    title="Comparação de embeddings - Query: 'Quero tirar uns dias de folga do trabalho'",
    color_discrete_map={True: "green", False: "red"}
)

fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(yaxis=dict(range=[0,1]), showlegend=True)

fig.show()


In [ ]:
import plotly.express as px

# --- exemplo: data com duas queries (substitua/expanda com seus resultados reais) ---
data_multi = [
    {"Query": "Quero tirar uns dias de folga do trabalho", "Modelo": "Gemini", "Top-1 retornado": "Qual é a política de férias da nossa empresa?", "Score Top-1": 0.831, "Acertou?": True},
    {"Query": "Quero tirar uns dias de folga do trabalho", "Modelo": "Multilingual-e5-large", "Top-1 retornado": "Qual é a política de férias da nossa empresa?", "Score Top-1": 0.856, "Acertou?": True},
    {"Query": "Quero tirar uns dias de folga do trabalho", "Modelo": "MiniLM", "Top-1 retornado": "Quero entender o processo de avaliação de performance.", "Score Top-1": 0.496, "Acertou?": False},
    {"Query": "Quero tirar uns dias de folga do trabalho", "Modelo": "BGE-large", "Top-1 retornado": "Qual é a política de férias da nossa empresa?", "Score Top-1": 0.646, "Acertou?": True},

    # Exemplo de segunda query (adicione quantas quiser)
    {"Query": "Como configuro o acesso à VPN?", "Modelo": "Gemini", "Top-1 retornado": "Como configuro o acesso à rede privada virtual (VPN)?", "Score Top-1": 0.78, "Acertou?": True},
    {"Query": "Como configuro o acesso à VPN?", "Modelo": "Multilingual-e5-large", "Top-1 retornado": "Como configuro o acesso à rede privada virtual (VPN)?", "Score Top-1": 0.80, "Acertou?": True},
    {"Query": "Como configuro o acesso à VPN?", "Modelo": "MiniLM", "Top-1 retornado": "Onde encontro o código de conduta da organização?", "Score Top-1": 0.44, "Acertou?": False},
    {"Query": "Como configuro o acesso à VPN?", "Modelo": "BGE-large", "Top-1 retornado": "Como configuro o acesso à rede privada virtual (VPN)?", "Score Top-1": 0.62, "Acertou?": True},
]

df_multi = pd.DataFrame(data_multi)

# --- gráfico: facet por Query, barras por Modelo ---
fig = px.bar(
    df_multi,
    x="Modelo",
    y="Score Top-1",
    color="Acertou?",
    facet_col="Query",
    text="Score Top-1",
    category_orders={"Modelo": df_multi["Modelo"].unique().tolist()},
    title="Comparação de Score Top-1 por Modelo — múltiplas queries",
    color_discrete_map={True: "green", False: "red"},
    labels={"Score Top-1": "Score Top-1", "Modelo": "Modelo"}
)

fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(yaxis=dict(range=[0,1]), showlegend=True)
# Ajuste de layout para exibir melhor múltiplos facets
fig.update_layout(uniformtext_minsize=8, uniformtext_mode='hide', bargap=0.2, height=400 + 200 * (df_multi["Query"].nunique()//2))

# Rotacionar os rótulos do eixo X
fig.update_xaxes(tickangle=45)

fig.show()

In [ ]:
import pandas as pd

# --- Exemplo de dataset para avaliação de modelos de embeddings ---
# Cada entrada contém:
#   Query: termo ou intenção da pesquisa
#   Modelo: nome do modelo de embeddings
#   Ranking: lista de documentos ordenados pelo score do modelo
#   Relevante: documento que consideramos correto para a query
rankings = [
    # Query 1: intenção de "tirar folga do trabalho"
    {"Query": "folga", "Modelo": "Gemini", "Ranking": [
        "Qual é a política de férias da nossa empresa?",
        "Preciso de um relatório de despesas de viagem.",
        "Quero entender o processo de avaliação de performance."
    ], "Relevante": "Qual é a política de férias da nossa empresa?"},

    {"Query": "folga", "Modelo": "Multilingual-e5-large", "Ranking": [
        "Qual é a política de férias da nossa empresa?",
        "Preciso de um relatório de despesas de viagem.",
        "Quero entender o processo de avaliação de performance."
    ], "Relevante": "Qual é a política de férias da nossa empresa?"},

    {"Query": "folga", "Modelo": "MiniLM", "Ranking": [
        "Quero entender o processo de avaliação de performance.",
        "Onde encontro o código de conduta da organização?",
        "Qual é a política de férias da nossa empresa?"
    ], "Relevante": "Qual é a política de férias da nossa empresa?"},

    {"Query": "folga", "Modelo": "BGE-large", "Ranking": [
        "Qual é a política de férias da nossa empresa?",
        "Preciso de um relatório de despesas de viagem.",
        "Onde encontro o código de conduta da organização?"
    ], "Relevante": "Qual é a política de férias da nossa empresa?"},

    # Query 2: intenção de "configurar acesso VPN"
    {"Query": "vpn", "Modelo": "Gemini", "Ranking": [
        "Como configuro o acesso à rede privada virtual (VPN)?",
        "Preciso de um relatório de despesas de viagem.",
        "Quero entender o processo de avaliação de performance."
    ], "Relevante": "Como configuro o acesso à rede privada virtual (VPN)?"},

    {"Query": "vpn", "Modelo": "Multilingual-e5-large", "Ranking": [
        "Como configuro o acesso à rede privada virtual (VPN)?",
        "Qual é a política de férias da nossa empresa?",
        "Preciso de um relatório de despesas de viagem."
    ], "Relevante": "Como configuro o acesso à rede privada virtual (VPN)?"},

    {"Query": "vpn", "Modelo": "MiniLM", "Ranking": [
        "Onde encontro o código de conduta da organização?",
        "Quero entender o processo de avaliação de performance.",
        "Qual é a política de férias da nossa empresa?"
    ], "Relevante": "Como configuro o acesso à rede privada virtual (VPN)?"},

    {"Query": "vpn", "Modelo": "BGE-large", "Ranking": [
        "Como configuro o acesso à rede privada virtual (VPN)?",
        "Onde encontro o código de conduta da organização?",
        "Quero entender o processo de avaliação de performance."
    ], "Relevante": "Como configuro o acesso à rede privada virtual (VPN)?"}
]

# --- Cria DataFrame para facilitar manipulação ---
df = pd.DataFrame(rankings)

# --- Função de avaliação de métricas ---
# Calcula:
#   Top@k: se o documento relevante aparece entre os k primeiros resultados (0 ou 1)
#   MRR: Reciprocal Rank, inverso da posição do documento relevante
def eval_metrics(row, k=3):
    try:
        # Posição do documento relevante (1-based)
        rank = row["Ranking"].index(row["Relevante"]) + 1
    except ValueError:
        rank = None  # caso não apareça no ranking

    topk = int(rank is not None and rank <= k)
    mrr = 1.0 / rank if rank is not None else 0.0
    return pd.Series({"Top@"+str(k): topk, "MRR": mrr})

# --- Aplica avaliação para cada linha (query×modelo) ---
results = df.join(df.apply(eval_metrics, axis=1))

# --- Agrega métricas por modelo ---
# Top@3 (acurácia) e MRR médio por modelo
metrics = results.groupby("Modelo")[["Top@3", "MRR"]].mean().reset_index()

# --- Resultado final ---
print(metrics)


                  Modelo  Top@3       MRR
0              BGE-large    1.0  1.000000
1                 Gemini    1.0  1.000000
2                 MiniLM    0.5  0.166667
3  Multilingual-e5-large    1.0  1.000000


In [ ]:
import pandas as pd
import plotly.express as px

# --- calcula Top@1, Top@3, Top@5 e MRR ---
def compute_topk_and_mrr(df, ks=[1,3,5]):
    for k in ks:
        def topk(row):
            try:
                rank = row["Ranking"].index(row["Relevante"]) + 1  # 1-based
            except ValueError:
                rank = None
            return int(rank is not None and rank <= k)
        df[f"Top@{k}"] = df.apply(topk, axis=1)

    # MRR
    def reciprocal_rank(row):
        try:
            rank = row["Ranking"].index(row["Relevante"]) + 1
            return 1.0 / rank
        except ValueError:
            return 0.0
    df["MRR"] = df.apply(reciprocal_rank, axis=1)

    # agregação por modelo
    metrics = df.groupby("Modelo")[[f"Top@{k}" for k in ks] + ["MRR"]].mean().reset_index()
    return df, metrics

# --- executa cálculo ---
df_per_query, metrics_combined = compute_topk_and_mrr(df, ks=[1,3,5])
print(metrics_combined)

# --- prepara para gráfico (long format) ---
metrics_long = metrics_combined.melt(id_vars="Modelo",
                                     value_vars=["Top@1","Top@3","Top@5","MRR"],
                                     var_name="Métrica", value_name="Valor")

# --- gráfico grouped bar ---
fig = px.bar(
    metrics_long,
    x="Modelo",
    y="Valor",
    color="Métrica",
    barmode="group",
    text="Valor",
    title="Comparação de Top-K Accuracy e MRR por Modelo",
    labels={"Valor": "Valor", "Modelo": "Modelo", "Métrica": "Métrica"}
)
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(yaxis=dict(range=[0,1]), uniformtext_minsize=8)
fig.show()


                  Modelo  Top@1  Top@3  Top@5       MRR
0              BGE-large    1.0    1.0    1.0  1.000000
1                 Gemini    1.0    1.0    1.0  1.000000
2                 MiniLM    0.0    0.5    0.5  0.166667
3  Multilingual-e5-large    1.0    1.0    1.0  1.000000


## Interpretação dos resultados do benchmark de embeddings

O gráfico **grouped bar** mostra a comparação de **Top@1, Top@3, Top@5 e MRR** entre os modelos testados: BGE-large, Gemini, MiniLM e Multilingual-e5-large.

### 1️⃣ Modelos de melhor desempenho
- **BGE-large, Gemini e Multilingual-e5-large**
  - **Top@1 = 1.0** → o documento mais relevante foi sempre o primeiro na lista.
  - **Top@3 e Top@5 = 1.0** → o documento relevante aparece consistentemente entre os primeiros.
  - **MRR = 1.0** → reforça que os modelos colocam o relevante no topo.
  - **Conclusão:** esses modelos capturam bem a semântica das queries, mesmo curtas ou em português.

### 2️⃣ Modelo com desempenho inferior
- **MiniLM**
  - **Top@1 = 0.0** → na maioria das queries, o documento relevante **não foi o primeiro**.
  - **Top@3 e Top@5 = 0.5** → o relevante aparece dentro do ranking, mas nem sempre em posições altas.
  - **MRR = 0.1667** → penaliza o modelo pela baixa posição do relevante.
  - **Conclusão:** MiniLM é mais leve e rápido, mas perde precisão em captura semântica, especialmente em queries curtas ou multilíngues.

### 3️⃣ Observações gerais
- O **Top@3 e Top@5** podem saturar rapidamente em rankings curtos, tornando-os menos discriminativos.
- O **MRR** é mais sensível à posição do relevante, permitindo diferenciar melhor o MiniLM.
- Para análises mais robustas, recomenda-se:
  - Aumentar o número de queries;
  - Aumentar o número de documentos por ranking;
  - Avaliar métricas como **Top-k accuracy** e **MRR** em conjunto com rerankers ou filtros semânticos.

> **Resumo visual:** No gráfico, cada grupo de barras representa um modelo. As cores correspondem às métricas (Top@1, Top@3, Top@5, MRR). Modelos que atingem o topo em todas as queries terão todas as barras no valor 1, enquanto modelos com menor desempenho terão barras mais baixas, evidenciando suas limitações.


In [ ]:
import pandas as pd
import random

# --- Configurações do teste ---
queries = [
    "folga", "vpn", "relatório de despesas", "avaliação de performance",
    "código de conduta", "acesso remoto", "benefícios", "ferias coletivas",
    "processo de contratação", "reembolso"
]

# Documentos possíveis (exemplo simplificado)
docs_pool = [
    "Qual é a política de férias da nossa empresa?",
    "Preciso de um relatório de despesas de viagem.",
    "Quero entender o processo de avaliação de performance.",
    "Onde encontro o código de conduta da organização?",
    "Como configuro o acesso à rede privada virtual (VPN)?",
    "Como solicitar benefícios adicionais?",
    "Quais são as férias coletivas previstas?",
    "Como funciona o processo de contratação?",
    "Qual é o procedimento de reembolso?",
    "Informações sobre acesso remoto seguro"
]

modelos = ["Gemini", "Multilingual-e5-large", "MiniLM", "BGE-large"]

# --- Gera rankings simulados ---
rankings = []
for q in queries:
    for m in modelos:
        # Escolhe um documento relevante aleatório para cada query
        relevante = random.choice(docs_pool)
        # Gera ranking aleatório garantindo que o relevante esteja na lista
        ranking = random.sample(docs_pool, 4)
        if relevante not in ranking:
            ranking.append(relevante)
        random.shuffle(ranking)

        rankings.append({
            "Query": q,
            "Modelo": m,
            "Ranking": ranking,
            "Relevante": relevante
        })

df = pd.DataFrame(rankings)

# --- Função para calcular Top@k e MRR ---
def compute_topk_and_mrr(df, ks=[1,3,5]):
    for k in ks:
        df[f"Top@{k}"] = df.apply(
            lambda row: int((row["Relevante"] in row["Ranking"][:k])), axis=1
        )
    df["MRR"] = df.apply(
        lambda row: 1.0/(row["Ranking"].index(row["Relevante"])+1) if row["Relevante"] in row["Ranking"] else 0.0,
        axis=1
    )
    metrics = df.groupby("Modelo")[[f"Top@{k}" for k in ks]+["MRR"]].mean().reset_index()
    return df, metrics

df_per_query, metrics_combined = compute_topk_and_mrr(df)
print(metrics_combined)


                  Modelo  Top@1  Top@3  Top@5       MRR
0              BGE-large    0.4    0.7    1.0  0.620000
1                 Gemini    0.1    0.6    1.0  0.400000
2                 MiniLM    0.4    0.8    1.0  0.600000
3  Multilingual-e5-large    0.1    0.6    1.0  0.373333


In [ ]:
import plotly.express as px

# --- transforma o DataFrame de métricas para long format ---
metrics_long = metrics_combined.melt(
    id_vars="Modelo",
    value_vars=["Top@1","Top@3","Top@5","MRR"],
    var_name="Métrica",
    value_name="Valor"
)

# --- gráfico grouped bar ---
fig = px.bar(
    metrics_long,
    x="Modelo",
    y="Valor",
    color="Métrica",
    barmode="group",
    text="Valor",
    title="Comparação de Top-K Accuracy e MRR por Modelo (Simulado)",
    labels={"Valor": "Valor", "Modelo": "Modelo", "Métrica": "Métrica"}
)

# --- formatação do gráfico ---
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig.update_layout(yaxis=dict(range=[0,1]), uniformtext_minsize=8)
fig.show()


# Glossário Visual de Métricas de Recuperação de Informação

Quando avaliamos modelos de embeddings, usamos métricas que mostram **onde o documento relevante aparece no ranking**. Vamos visualizar cada uma delas.

---

## **Top@1** ✅
- **O que mede:** Se o documento relevante está **em 1º lugar**.
- **Visualização:**
```

1️⃣ Documento relevante ✔
2️⃣ Outro documento
3️⃣ Outro documento

```
- **Interpretação do valor:**  
  - 1 → sucesso (relevante em 1º)  
  - 0 → falha (relevante não está em 1º)

---

## **Top@3** 🟢
- **O que mede:** Se o documento relevante está **entre os 3 primeiros**.
- **Visualização:**
```

1️⃣ Outro documento
2️⃣ Documento relevante ✔
3️⃣ Outro documento

```
- **Interpretação do valor:**  
  - 1 → relevante aparece no Top 3  
  - 0 → não aparece no Top 3

---

## **Top@5** 🔵
- **O que mede:** Se o documento relevante está **entre os 5 primeiros**.
- **Visualização:**
```

1️⃣ Outro documento
2️⃣ Outro documento
3️⃣ Documento relevante ✔
4️⃣ Outro documento
5️⃣ Outro documento

```
- **Interpretação do valor:**  
  - 1 → relevante aparece no Top 5  
  - 0 → não aparece no Top 5

---

## **MRR (Mean Reciprocal Rank)** 💡
- **O que mede:** A **posição média do documento relevante**, penalizando posições mais baixas.
- **Cálculo rápido:**  
\[
\text{RR} = \frac{1}{\text{posição do relevante}}
\]  
  - 1º lugar → RR = 1.0  
  - 2º lugar → RR = 0.5  
  - 5º lugar → RR = 0.2
- **Para várias queries:** Média dos RR → **MRR**
- **Visualização simplificada:**
```

Ranking: [Doc A, Doc B ✔, Doc C, Doc D]
Posição do relevante: 2
RR = 1/2 = 0.5

```

---

## **Resumo rápido**
| Métrica | Faixa | O que mostra |
|---------|-------|--------------|
| Top@1   | 0–1   | Relevante em 1º? |
| Top@3   | 0–1   | Relevante entre os 3 primeiros? |
| Top@5   | 0–1   | Relevante entre os 5 primeiros? |
| MRR     | 0–1   | Posição média do relevante (quanto maior, melhor) |

> 💡 **Dica de estudo:**  
> - **Top@k** → simples e intuitivo; ótimo para análises rápidas.  
> - **MRR** → mais sensível à posição; ideal para rankings longos ou consultas críticas.

---

### ✅ Conclusão
- Use **Top@k** para avaliar se o modelo acerta o topo do ranking.  
- Use **MRR** para avaliar a posição exata do relevante dentro do ranking.  
- Juntas, essas métricas ajudam a entender **precisão e qualidade do ranking** de cada modelo.


##3.2 Caching de Embeddings: Economia e Velocidade

In [ ]:
from langchain_community.storage.file import LocalFileStore
from langchain.embeddings import CacheBackedEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings


store = LocalFileStore("./cache/")

embedder_principal = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001"
)

cache_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embedder_principal,
    store,
    namespace="gemini_cache"
)

# exemplo
texto = "Este é um teste de embeddings."
vector = cache_embeddings.embed_query(texto)
vector[:5]

ModuleNotFoundError: No module named 'langchain_community.storage.file'

In [ ]:
textos_para_cache = ["Olá, mundo!", "Testando o cache de embeddings.", "Olá, mundo!"]

start_time = time.time()
embeddings_result_1 = cache_embeddings.embed_documents(textos_para_cache)
end_time = time.time()

print(f"  - Tempo de execução: {end_time - start_time:.4f} segundos.")

NameError: name 'cache_embeddings' is not defined

##3. Batch Processing para Indexação em Larga Escala

>Ao indexar milhares de documentos, processá-los em lotes (batches) é fundamental. Modelos locais, especialmente, se beneficiam enormemente disso.

In [ ]:
documentos_grandes = [f"Este é o documento de teste número {i}." for i in range(1000)]

bge_embedder = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={"device": "gpu"},
    encode_kwargs={"normalize_embeddings": True}
)

batch_sizes = [1, 32, 64, 128]

22:58:15 | INFO    | Load pretrained SentenceTransformer: BAAI/bge-large-en-v1.5


RuntimeError: Expected one of cpu, cuda, ipu, xpu, mkldnn, opengl, opencl, ideep, hip, ve, fpga, maia, xla, lazy, vulkan, mps, meta, hpu, mtia, privateuseone device type at start of device string: gpu

In [ ]:
for batch_size in batch_sizes:
    start_time = time.time()
    num_batches = len(documentos_grandes) // batch_size
    tempo_estimado = num_batches * (0.1 * batch_size) + (len(documentos_grandes) % batch_size) * 0.1
    tempo_real = bge_embedder.client.encode(documentos_grandes, batch_size=batch_size)
    end_time = time.time()

    print(f"  - Batch Size: {batch_size:<4} -> Tempo: {end_time - start_time:.2f}s")

#04 Pipeline para Dados Complexos